# package imports

In [ ]:
from scpviz import pAnnData as pAnnData
from scpviz import plotting as scplt
from scpviz import utils as scutils
import scanpy as sc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context='paper', style='ticks')

import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

# Combined data files

The same report file is used in Fig 2, Fig 4 and Fig 5 - `pdata_all_region_filtered`

In [ ]:
import os

pdata_all_region = pAnnData.import_data(source_type='diann',report_file = 'data/2505_rna_prot_full/report.tsv')

# data clean-up | uses file_annotation.csv to help with annotations

file_annotation = pd.read_csv('abc_analysis/file_annotation.csv')

file_annotation["parsed_filename"] = (
    file_annotation["RAW FILE"]
    .apply(os.path.basename)
    .str.replace(".raw", "", regex=False)
)

summary = pdata_all_region.summary

summary_files = set(summary.index)
annotation_files = set(file_annotation["parsed_filename"])
missing_in_summary = annotation_files - summary_files
missing_in_annotation = summary_files - annotation_files
pdata_all_region_filtered = pdata_all_region.filter_sample(exclude_file_list=list(missing_in_annotation))

fa_sub = file_annotation[["parsed_filename", "File Name", "Grouping", "Sub grouping", "Region", "Batch"]]
fa_sub = fa_sub.set_index("parsed_filename")
fa_aligned = fa_sub.reindex(pdata_all_region_filtered.summary.index)
cols_to_add = ["File Name", "Grouping", "Sub grouping", "Region", "Batch"]
pdata_all_region_filtered.summary[cols_to_add] = fa_aligned[cols_to_add]

pdata_all_region_filtered.update_summary()

pd.DataFrame(pdata_all_region_filtered.summary)

In [ ]:
pd.DataFrame(pdata_all_region.summary)

# Fig 1 - size

In [ ]:
import re
import pandas as pd
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def build_peptide_gravy_df(acc_list, pdata, group):
    prot_names = pd.Index(pdata.prot.var_names.astype(str))
    pep_names = pd.Index(pdata.pep.var_names.astype(str))

    rows = []

    for acc in acc_list:
        acc = str(acc).strip()

        if acc not in prot_names:
            continue

        i = prot_names.get_loc(acc)
        pep_idx = pdata.rs[i].nonzero()[1]

        for pep_id in pep_names[pep_idx]:
            pep_clean = str(pep_id)

            # examples:
            # [R].MQHNLEQQIQAR.[N] -> MQHNLEQQIQAR
            # [R].MQHNLEQQIQAR.[N] MOD:1xOxidation [M1] -> MQHNLEQQIQAR
            pep_clean = pep_clean.split(" MOD:")[0]
            pep_clean = re.sub(r"^\[[A-Z]\]\.", "", pep_clean)
            pep_clean = re.sub(r"\.\[[A-Z]\]$", "", pep_clean)
            pep_clean = re.sub(r"\d+$", "", pep_clean)

            try:
                gravy = ProteinAnalysis(pep_clean).gravy()
            except Exception:
                gravy = pd.NA

            rows.append({
                "accession": acc,
                "peptide_id": pep_id,
                "peptide_clean": pep_clean,
                "gravy": gravy,
                "group": group,
            })

    return pd.DataFrame(rows)

## size

In [ ]:
pdata = pAnnData.import_data(source_type = 'diann', report_file = 'data/2411_size/report.tsv', obs_columns = ['date','gradient','size','batch','replicate'], prot_value = 'PG.Quantity')

# change size = 10k2 to 10k
pdata.summary['size'] = pdata.summary['size'].replace('10k2','10k')
pdata.update_summary()

In [ ]:
pdata_filter = pdata.filter_sample(min_prot=1000)
pdata_filter = pdata_filter.filter_prot_significant()
pdata_filter = pdata_filter.filter_sample(condition = "(size == 'sc') or (protein_quant > 0.35)")
pdata_filter = pdata_filter.filter_sample(condition = "not(size == '2k' and protein_count > 3000)") # 2000 outlier

pdata_filter = pdata_filter.filter_sample(exclude_file_list=['20240425_Aur60minDIA_10k2_T2_01'])

pd.DataFrame(pdata_filter.summary)

## fixed vs frozen

In [ ]:
obs_columns = ['sample', 'amount', 'type']

pdata_ff = pAnnData.import_proteomeDiscoverer(prot_file = 'data/MouseFF_100ng_FixedFrozen_43minDDA_20230623_Proteins.txt', pep_file='data/MouseFF_100ng_FixedFrozen_43minDDA_20230623_PeptideGroups.txt', obs_columns=obs_columns)

In [ ]:
pdata_ff.summary['type'] = pdata_ff.summary['type'].replace({'fixed': 'Fixed', 'frozen': 'Frozen'})
pdata_ff.update_summary()

In [ ]:
ff_upset = scutils.get_upset_contents(pdata_ff, classes = ['type'])

pdata_fixed_only = pdata_ff.filter_sample(condition = 'type == "Fixed"')
pdata_frozen_only = pdata_ff.filter_sample(condition = 'type == "Frozen"')

fixed_only = scutils.get_upset_query(ff_upset, present='Fixed', absent='Frozen', fetch_uniprot=False, pdata=pdata_ff)
frozen_only = scutils.get_upset_query(ff_upset, present='Frozen', absent='Fixed', fetch_uniprot=False, pdata=pdata_ff)

fixed_all = scutils.get_upset_query(ff_upset, present='Fixed', absent=[], fetch_uniprot=False, pdata=pdata_ff)
frozen_all = scutils.get_upset_query(ff_upset, present='Frozen', absent=[], fetch_uniprot=False, pdata=pdata_ff)

# export ab_only and pbs_only to csv

In [ ]:
fixed_acc_list = (
    fixed_all["accession"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s != ",")].tolist()
)
frozen_acc_list = (
    frozen_all["accession"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s != ",")].tolist()
)

In [ ]:
fixed_gravy_df = pdata_ff.get_peptide_properties(
    properties=["gravy"],
    accessions=fixed_acc_list,
    return_copy=True,  # don't write to .pep.var unless you want that
)
fixed_gravy_df["group"] = "fixed"
frozen_gravy_df = pdata_ff.get_peptide_properties(
    properties=["gravy"],
    accessions=frozen_acc_list,
    return_copy=True,
)
frozen_gravy_df["group"] = "frozen"

peptide_gravy_df = pd.concat(
    [fixed_gravy_df, frozen_gravy_df],
    ignore_index=True,
)

gravy_df = peptide_gravy_df.dropna(subset=["gravy"]).copy()
gravy_df["gravy"] = gravy_df["gravy"].astype(float)
# one row per unique peptide sequence within each group
gravy_df_unique = gravy_df.drop_duplicates(subset=["group", "peptide_sequence"]).copy()

fixed_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "fixed", "gravy"]
frozen_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "frozen", "gravy"]

fixed_seqs = set(gravy_df_unique.loc[gravy_df_unique["group"] == "fixed", "peptide_sequence"])
frozen_seqs = set(gravy_df_unique.loc[gravy_df_unique["group"] == "frozen", "peptide_sequence"])

shared_seqs = fixed_seqs & frozen_seqs
gravy_df_unique["overlap"] = gravy_df_unique["peptide_sequence"].isin(shared_seqs)
gravy_df_nonoverlap = gravy_df_unique.loc[~gravy_df_unique["overlap"]].copy()

fixed_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "fixed", "gravy"]
frozen_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "frozen", "gravy"]
fixed_gravy_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == "fixed", "gravy"]
frozen_gravy_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == "frozen", "gravy"]

from scipy.stats import ranksums

statistic, p_value = ranksums(fixed_gravy, frozen_gravy)
_, p_value_non = ranksums(fixed_gravy_non, frozen_gravy_non)
print(f"Fixed n = {len(fixed_gravy)} (non-overlap n = {len(fixed_gravy_non)})")
print(f"Frozen n = {len(frozen_gravy)} (non-overlap n = {len(frozen_gravy_non)})")
print(f"Shared peptide sequences = {len(shared_seqs)}")
print(f"p-value (all unique) = {p_value:.2e}")
print(f"p-value (non-overlap) = {p_value_non:.2e}")

In [ ]:
from scipy.stats import gaussian_kde

palette_ab = sns.color_palette("Paired")[0:2]
colors = [palette_ab[1], palette_ab[0]]
groups = ["fixed", "frozen"]
labels = ["Fixed", "Frozen"]

fig, ax = plt.subplots(figsize=(4, 2.5))

x_min = gravy_df_unique["gravy"].min()
x_max = gravy_df_unique["gravy"].max()
x_grid = np.linspace(x_min, x_max, 500)

lvl_max = 0
for i, group in enumerate(groups):
    vals_all = gravy_df_unique.loc[gravy_df_unique["group"] == group, "gravy"].to_numpy()
    vals_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == group, "gravy"].to_numpy()
    sign = (-1) ** i

    # scale by n
    kde_all = gaussian_kde(vals_all)(x_grid) * len(vals_all)
    ax.fill_between(
        x_grid, kde_all * sign,
        color=colors[i], alpha=0.35, linewidth=0,
        label=f"{labels[i]} (all)",
    )

    # scale by n
    if len(vals_non) > 1 and np.std(vals_non) > 0:
        kde_non = gaussian_kde(vals_non)(x_grid) * len(vals_non) * 3
        ax.fill_between(
            x_grid, kde_non * sign,
            color=colors[i], alpha=1.0, linewidth=0, zorder=2,
            label=f"{labels[i]} (non-overlap)",
        )
        lvl_max = max(lvl_max, np.max(kde_non))
        
    mode_x = x_grid[np.argmax(kde_all)]
    ax.vlines(mode_x, 0, np.max(kde_all) * sign, color=colors[i], linestyle="--", linewidth=1.2)
    lvl_max = max(lvl_max, np.max(kde_all))

ax.text(
    0.15, 0.85,
    f"p (all) = {'%.2E' % Decimal(p_value)}",
    fontsize=8,
    transform=fig.transFigure,
)
ax.text(
    0.15, 0.8,
    f"p (non-overlap) = {'%.2E' % Decimal(p_value_non)}",
    fontsize=8,
    transform=fig.transFigure,
)
ax.text(
    0.55, 0.85,
    f"Fixed | n = {len(fixed_gravy)} / {len(fixed_gravy_non)} non-overlap",
    fontsize=8,
    transform=fig.transFigure,
    color=colors[0],
)
ax.text(
    0.55, 0.8,
    f"Frozen | n = {len(frozen_gravy)} / {len(frozen_gravy_non)} non-overlap",
    fontsize=8,
    transform=fig.transFigure,
    color=colors[1],
)
ax.set_ylim(-lvl_max * 1.05, lvl_max * 1.05)
yticks = ax.get_yticks()
ax.set_yticks(yticks)
ax.set_yticklabels([int(abs(y)) for y in yticks])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Peptide GRAVY score")
ax.set_ylabel("Number of peptides")
ax.legend(frameon=False, fontsize=7)

plt.savefig('MANUSCRIPT_revision/REVISION_FigS1-GRAVY_FF.png', dpi=450, bbox_inches='tight')

## marker - sc

In [ ]:
pdata_fig1_supplementary = pAnnData.import_data(source_type='diann', report_file = 'data/2512_Ab-HCR/report.parquet')

In [ ]:
pdata_fig1_supplementary.summary = scutils.parse_filename_index(pdata_fig1_supplementary.summary, obs_columns=['date','acquisition','size','buffer','well_position'], condition='parsingType == "5-tokens"')
pdata_fig1_supplementary.summary = scutils.parse_filename_index(pdata_fig1_supplementary.summary, obs_columns=['date','acquisition','sample_id','size','confirmation','thickness','type','organism','region','well_position'], condition='parsingType == "10-tokens"')

mapping = {"800": "sc",}
pdata_fig1_supplementary.summary["size"] = pdata_fig1_supplementary.summary["size"].replace(mapping)
pdata_fig1_supplementary.update_summary()

pdata_fig1_supplementary_sc = pdata_fig1_supplementary.filter_sample(condition="size=='sc'")

# for dataset with single cell stained vs non-stained
# ab stained is april, pbs stained is november
summary = pdata_fig1_supplementary_sc.summary
summary.loc[summary["date"].astype(str).str.startswith("202404"), "type"] = "IF"
summary.loc[summary["date"].astype(str).str.startswith("202411"), "type"] = "Control"

pdata_fig1_supplementary_sc.summary = summary
pdata_fig1_supplementary_sc.update_summary()

pd.DataFrame(pdata_fig1_supplementary_sc.summary)

#### IF (ab vs pbs)

In [ ]:
pdata_ab_sc = pdata_fig1_supplementary_sc.filter_sample(condition = 'type in ("IF", "Control")')
pdata_ab_sc = pdata_ab_sc.filter_sample(min_prot = 1000)
pdata_ab_sc = pdata_ab_sc.filter_prot_significant()

pdata_ab_sc.summary.type.value_counts()

In [ ]:
ab_upset = scutils.get_upset_contents(pdata_ab_sc, classes = ['type'])

pdata_ab_only = pdata_ab_sc.filter_sample(query_mode=True, values = 'type == "IF"', return_copy=True)
pdata_pbs_only = pdata_ab_sc.filter_sample(query_mode=True, values ='type == "Control"', return_copy=True)

ab_only = scutils.get_upset_query(ab_upset, present='IF', absent='Control', fetch_uniprot=False, pdata=pdata_ab_sc)
pbs_only = scutils.get_upset_query(ab_upset, present='Control', absent='IF', fetch_uniprot=False, pdata=pdata_ab_sc)

ab_all = scutils.get_upset_query(ab_upset, present='IF', absent=[], fetch_uniprot=False, pdata=pdata_ab_sc)
pbs_all = scutils.get_upset_query(ab_upset, present='Control', absent=[], fetch_uniprot=False, pdata=pdata_ab_sc)

In [ ]:
ab_acc_list = (
    ab_all["accession"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s != ",")].tolist()
)

pbs_acc_list = (
    pbs_all["accession"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s != ",")].tolist()
)

ab_gravy_df = pdata_ab_sc.get_peptide_properties(
    properties=["gravy"],
    accessions=ab_acc_list,
    return_copy=True,  # don't write to .pep.var unless you want that
)
ab_gravy_df["group"] = "ab"

pbs_gravy_df = pdata_ab_sc.get_peptide_properties(
    properties=["gravy"],
    accessions=pbs_acc_list,
    return_copy=True,
)
pbs_gravy_df["group"] = "pbs"

peptide_gravy_df = pd.concat(
    [ab_gravy_df, pbs_gravy_df],
    ignore_index=True,
)

gravy_df = peptide_gravy_df.dropna(subset=["gravy"]).copy()
gravy_df["gravy"] = gravy_df["gravy"].astype(float)
# one row per unique peptide sequence within each group
gravy_df_unique = gravy_df.drop_duplicates(subset=["group", "peptide_sequence"]).copy()

ab_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "ab", "gravy"]
pbs_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "pbs", "gravy"]

ab_seqs = set(gravy_df_unique.loc[gravy_df_unique["group"] == "ab", "peptide_sequence"])
pbs_seqs = set(gravy_df_unique.loc[gravy_df_unique["group"] == "pbs", "peptide_sequence"])

shared_seqs = ab_seqs & pbs_seqs
gravy_df_unique["overlap"] = gravy_df_unique["peptide_sequence"].isin(shared_seqs)
gravy_df_nonoverlap = gravy_df_unique.loc[~gravy_df_unique["overlap"]].copy()

ab_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "ab", "gravy"]
pbs_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "pbs", "gravy"]
ab_gravy_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == "ab", "gravy"]
pbs_gravy_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == "pbs", "gravy"]

from scipy.stats import ranksums

statistic, p_value = ranksums(ab_gravy, pbs_gravy)
_, p_value_non = ranksums(ab_gravy_non, pbs_gravy_non)
print(f"Ab n = {len(ab_gravy)} (non-overlap n = {len(ab_gravy_non)})")
print(f"PBS n = {len(pbs_gravy)} (non-overlap n = {len(pbs_gravy_non)})")
print(f"Shared peptide sequences = {len(shared_seqs)}")
print(f"p-value (all unique) = {p_value:.2e}")
print(f"p-value (non-overlap) = {p_value_non:.2e}")

In [ ]:
from scipy.stats import gaussian_kde

palette_ab = sns.color_palette("Paired")[2:4]
colors = [palette_ab[1], palette_ab[0]]
groups = ["ab", "pbs"]
labels = ["IF", "Control"]

fig, ax = plt.subplots(figsize=(4, 2.5))

x_min = gravy_df_unique["gravy"].min()
x_max = gravy_df_unique["gravy"].max()
x_grid = np.linspace(x_min, x_max, 500)

lvl_max = 0

for i, group in enumerate(groups):
    vals_all = gravy_df_unique.loc[gravy_df_unique["group"] == group, "gravy"].to_numpy()
    vals_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == group, "gravy"].to_numpy()
    sign = (-1) ** i

    # scale by n
    kde_all = gaussian_kde(vals_all)(x_grid) * len(vals_all)
    ax.fill_between(
        x_grid, kde_all * sign,
        color=colors[i], alpha=0.35, linewidth=0,
        label=f"{labels[i]} (all)",
    )

    # scale by n
    if len(vals_non) > 1 and np.std(vals_non) > 0:
        kde_non = gaussian_kde(vals_non)(x_grid) * len(vals_non) * 3
        ax.fill_between(
            x_grid, kde_non * sign,
            color=colors[i], alpha=1.0, linewidth=0, zorder=2,
            label=f"{labels[i]} (non-overlap)",
        )
        lvl_max = max(lvl_max, np.max(kde_non))

    mode_x = x_grid[np.argmax(kde_all)]
    ax.vlines(mode_x, 0, np.max(kde_all) * sign, color=colors[i], linestyle="--", linewidth=1.2)
    lvl_max = max(lvl_max, np.max(kde_all))

ax.text(
    0.15, 0.85,
    f"p (all) = {'%.2E' % Decimal(p_value)}",
    fontsize=8,
    transform=fig.transFigure,
)
ax.text(
    0.15, 0.8,
    f"p (non-overlap) = {'%.2E' % Decimal(p_value_non)}",
    fontsize=8,
    transform=fig.transFigure,
)
ax.text(
    0.55, 0.85,
    f"IF | n = {len(ab_gravy)} / {len(ab_gravy_non)} non-overlap",
    fontsize=8,
    transform=fig.transFigure,
    color=colors[0],
)
ax.text(
    0.55, 0.8,
    f"Control | n = {len(pbs_gravy)} / {len(pbs_gravy_non)} non-overlap",
    fontsize=8,
    transform=fig.transFigure,
    color=colors[1],
)
ax.set_ylim(-lvl_max * 1.05, lvl_max * 1.05)
yticks = ax.get_yticks()
ax.set_yticks(yticks)
ax.set_yticklabels([int(abs(y)) for y in yticks])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Peptide GRAVY score")
ax.set_ylabel("Number of peptides")
ax.legend(frameon=False, fontsize=7)
plt.savefig('MANUSCRIPT_revision/REVISION_FigS1-GRAVY_sc-IF.png', dpi=450, bbox_inches='tight')

In [ ]:
pdata_ab_sc_processed = pdata_ab_sc.copy()
pdata_ab_sc_processed = pdata_ab_sc_processed.filter_prot_found(min_ratio=0.4, group = ['type'], match_any=True)
pdata_ab_sc_processed = pdata_ab_sc_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_ab_sc_norm = pdata_ab_sc_processed.copy()

In [ ]:
classes='type'
values=['IF','Control']

colors = sns.color_palette("Paired")[2:4]
color_dict = dict(zip(['downregulated', 'upregulated'], colors))

fig, ax = plt.subplots(1,1, figsize = (3,3))
scplt.plot_volcano(ax, pdata_ab_sc_norm, classes=classes, values=values, log2fc=1, pval=0.05, color=color_dict, correct_fdr=True,
                   group_annot_kwargs={"pos": {"group1_xy": (0.98, 1.08), "group2_xy": (0.02, 1.08)}},)
plt.savefig("MANUSCRIPT/MANUSCRIPT_FigS1-sc_IF_volcano.svg", bbox_inches="tight")
df_ab_norm = pdata_ab_sc_norm.stats["IF vs Control"]
df_ab_norm.to_csv('MANUSCRIPT_revision/REVISION_FigS1_DE-Ab-adjP.csv')

#### HCR (hcr vs pbs)

In [ ]:
# obs_columns
pdata_hcr_sc = pdata_fig1_supplementary_sc.filter_sample(condition = 'type in ("GADHCR", "Control")')
pdata_hcr_sc = pdata_hcr_sc.filter_sample(min_prot = 1000)
pdata_hcr_sc = pdata_hcr_sc.filter_prot_significant()

mapping = {
    "GADHCR": "HCR",
}
pdata_hcr_sc.summary["type"] = pdata_hcr_sc.summary["type"].replace(mapping)
pdata_hcr_sc.update_summary()

pdata_hcr_sc.summary.type.value_counts()

In [ ]:
ab_upset = scutils.get_upset_contents(pdata_hcr_sc, classes = ['type'])

pdata_HCR_only = pdata_hcr_sc.filter_sample(query_mode=True, values='type == "HCR"', return_copy=True)
pdata_control_only = pdata_hcr_sc.filter_sample(query_mode=True, values='type == "Control"', return_copy=True)


HCR_only = scutils.get_upset_query(ab_upset, present='HCR', absent='Control', fetch_uniprot=False, pdata=pdata_hcr_sc)
Control_only = scutils.get_upset_query(ab_upset, present='Control', absent='HCR', fetch_uniprot=False, pdata=pdata_hcr_sc)

hcr_all = scutils.get_upset_query(ab_upset, present='HCR', absent=[], fetch_uniprot=False, pdata=pdata_hcr_sc)
Control_all = scutils.get_upset_query(ab_upset, present='Control', absent=[], fetch_uniprot=False, pdata=pdata_hcr_sc)

# export ab_only and pbs_only to csv

In [ ]:
hcr_acc_list = (
    hcr_all["accession"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s != ",")].tolist()
)

control_acc_list = (
    Control_all["accession"].dropna().astype(str).str.strip()
    .loc[lambda s: (s != "") & (s != ",")].tolist()
)

hcr_gravy_df = pdata_hcr_sc.get_peptide_properties(
    properties=["gravy"],
    accessions=hcr_acc_list,
    return_copy=True,  # don't write to .pep.var unless you want that
)
hcr_gravy_df["group"] = "hcr"

control_gravy_df = pdata_hcr_sc.get_peptide_properties(
    properties=["gravy"],
    accessions=control_acc_list,
    return_copy=True,
)
control_gravy_df["group"] = "control"

peptide_gravy_df = pd.concat(
    [hcr_gravy_df, control_gravy_df],
    ignore_index=True,
)

gravy_df = peptide_gravy_df.dropna(subset=["gravy"]).copy()
gravy_df["gravy"] = gravy_df["gravy"].astype(float)
# one row per unique peptide sequence within each group
gravy_df_unique = gravy_df.drop_duplicates(subset=["group", "peptide_sequence"]).copy()

hcr_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "hcr", "gravy"]
control_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "control", "gravy"]

hcr_seqs = set(gravy_df_unique.loc[gravy_df_unique["group"] == "hcr", "peptide_sequence"])
control_seqs = set(gravy_df_unique.loc[gravy_df_unique["group"] == "control", "peptide_sequence"])

shared_seqs = hcr_seqs & control_seqs
gravy_df_unique["overlap"] = gravy_df_unique["peptide_sequence"].isin(shared_seqs)
gravy_df_nonoverlap = gravy_df_unique.loc[~gravy_df_unique["overlap"]].copy()

hcr_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "hcr", "gravy"]
control_gravy = gravy_df_unique.loc[gravy_df_unique["group"] == "control", "gravy"]
hcr_gravy_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == "hcr", "gravy"]
control_gravy_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == "control", "gravy"]

from scipy.stats import ranksums

statistic, p_value = ranksums(hcr_gravy, control_gravy)
_, p_value_non = ranksums(hcr_gravy_non, control_gravy_non)
print(f"HCR n = {len(hcr_gravy)} (non-overlap n = {len(hcr_gravy_non)})")
print(f"Control n = {len(control_gravy)} (non-overlap n = {len(control_gravy_non)})")
print(f"Shared peptide sequences = {len(shared_seqs)}")
print(f"p-value (all unique) = {p_value:.2e}")
print(f"p-value (non-overlap) = {p_value_non:.2e}")

In [ ]:
from scipy.stats import gaussian_kde

palette_ab = sns.color_palette("Paired")[4:6]
colors = [palette_ab[1], palette_ab[0]]
groups = ["hcr", "control"]
labels = ["HCR", "Control"]

fig, ax = plt.subplots(figsize=(4, 2.5))

x_min = gravy_df_unique["gravy"].min()
x_max = gravy_df_unique["gravy"].max()
x_grid = np.linspace(x_min, x_max, 500)

lvl_max = 0

for i, group in enumerate(groups):
    vals_all = gravy_df_unique.loc[gravy_df_unique["group"] == group, "gravy"].to_numpy()
    vals_non = gravy_df_nonoverlap.loc[gravy_df_nonoverlap["group"] == group, "gravy"].to_numpy()
    sign = (-1) ** i

    # scale by n
    kde_all = gaussian_kde(vals_all)(x_grid) * len(vals_all)
    ax.fill_between(
        x_grid, kde_all * sign,
        color=colors[i], alpha=0.35, linewidth=0,
        label=f"{labels[i]} (all)",
    )

    # scale by n
    if len(vals_non) > 1 and np.std(vals_non) > 0:
        kde_non = gaussian_kde(vals_non)(x_grid) * len(vals_non) * 3
        ax.fill_between(
            x_grid, kde_non * sign,
            color=colors[i], alpha=1.0, linewidth=0, zorder=2,
            label=f"{labels[i]} (non-overlap)",
        )
        lvl_max = max(lvl_max, np.max(kde_non))

    mode_x = x_grid[np.argmax(kde_all)]
    ax.vlines(mode_x, 0, np.max(kde_all) * sign, color=colors[i], linestyle="--", linewidth=1.2)
    lvl_max = max(lvl_max, np.max(kde_all))

ax.text(
    0.15, 0.85,
    f"p (all) = {'%.2E' % Decimal(p_value)}",
    fontsize=8,
    transform=fig.transFigure,
)
ax.text(
    0.15, 0.8,
    f"p (non-overlap) = {'%.2E' % Decimal(p_value_non)}",
    fontsize=8,
    transform=fig.transFigure,
)
ax.text(
    0.55, 0.85,
    f"HCR | n = {len(hcr_gravy)} / {len(hcr_gravy_non)} non-overlap",
    fontsize=8,
    transform=fig.transFigure,
    color=colors[0],
)
ax.text(
    0.55, 0.8,
    f"Control | n = {len(control_gravy)} / {len(control_gravy_non)} non-overlap",
    fontsize=8,
    transform=fig.transFigure,
    color=colors[1],
)

ax.set_ylim(-lvl_max * 1.05, lvl_max * 1.05)
yticks = ax.get_yticks()
ax.set_yticks(yticks)
ax.set_yticklabels([int(abs(y)) for y in yticks])
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Peptide GRAVY score")
ax.set_ylabel("Number of peptides")
ax.legend(frameon=False, fontsize=7)

plt.savefig('MANUSCRIPT_revision/REVISION_FigS1-GRAVY_sc-HCR.png', dpi=450, bbox_inches='tight')

In [ ]:
pdata_hcr_sc_processed = pdata_hcr_sc.copy()
pdata_hcr_sc_processed = pdata_hcr_sc_processed.filter_prot_found(min_ratio=0.4, group = ['type'], match_any=True)
pdata_hcr_sc_processed = pdata_hcr_sc_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_hcr_sc_norm = pdata_hcr_sc_processed.copy()

In [ ]:
classes='type'
values=['HCR','Control']

colors = sns.color_palette("Paired")[4:6]
color_dict = dict(zip(['downregulated', 'upregulated'], colors))

fig, ax = plt.subplots(1,1, figsize = (3.5,3.5))
scplt.plot_volcano(ax, pdata_hcr_sc_norm, classes=classes, values=values, log2fc=1, pval=0.05, color=color_dict, correct_fdr=True)
df_hcr_norm = pdata_hcr_sc_norm.stats["HCR vs Control"]
df_hcr_norm.to_csv('MANUSCRIPT_revision/REVISION_FigS1_DE-HCR-adjP.csv')

# Fig 2 - region

## Cortex vs snpc

### pre-processing

In [ ]:
pdata_fig2_region_filter = pdata_all_region_filtered.filter_prot_significant()
pdata_fig2_region_filter = pdata_fig2_region_filter.filter_sample(condition="Grouping != 'ast'")
pdata_fig2_region_filter = pdata_fig2_region_filter.filter_sample(min_prot=1000)

pdata_fig2_region_filter.summary["region"] = pdata_fig2_region_filter.summary["Grouping"].replace({
    "snpc": "SNpc",
    "ctx": "Cortex"
})
pdata_fig2_region_filter.update_summary()

pdata_fig2_region_processed = pdata_fig2_region_filter.copy()
pdata_fig2_region_processed = pdata_fig2_region_processed.filter_prot_found(min_ratio=0.4, group = ['Grouping'], match_any=True)
pdata_fig2_region_processed = pdata_fig2_region_processed.filter_prot(valid_genes=True, unique_profiles=True)

In [ ]:
pdata_fig2_region_norm = pdata_fig2_region_processed.copy()
pdata_fig2_region_norm.normalize(method='directlfq')

### plots

In [ ]:
cns_color={
    "Cortex": "#D19DCB",
    "SNpc": "#85BE9E"
}

In [ ]:
pdata_fig2_region_norm.summary

In [ ]:
fig, ax = plt.subplots(1,1, figsize = (3,3))
ax, pca = scplt.plot_pca(ax, pdata_fig2_region_norm, classes='region', s=20, alpha=.8, return_fit=True, force=True, cmap=cns_color, add_ellipses=False)
scplt.shift_legend(ax)


ax.set_xlabel(ax.get_xlabel(), fontsize=13, labelpad=4)
ax.set_ylabel(ax.get_ylabel(), fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.show()

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection='3d')
ax = scplt.plot_pca(ax, pdata_fig2_region_norm, classes=['region'], force=True, plot_pc = [1,2,3], cmap=cns_color)
zlab = ax.get_zlabel()
ax.set_zlabel("")
ax.text2D(1.05, 0.55, zlab,
          transform=ax.transAxes, rotation=90,
          ha='left', va='center', fontsize=9.5)


plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize = (3,3))

edge_color={
    "Apr-25":"k",
    "Apr-24":"none",
    "Aug-24":"none",
    "Jun-24":"none",
    "Nov-24":"none"
}

ax, pca = scplt.plot_pca(ax, pdata_fig2_region_norm, edge_color='Batch', color='region', cmap=cns_color, edge_cmap=edge_color, s=20, alpha=.8, return_fit=True, force=True, add_ellipses=False)


ax.set_xlabel(ax.get_xlabel(), fontsize=13, labelpad=4)
ax.set_ylabel(ax.get_ylabel(), fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

scplt.shift_legend(ax)
plt.show()

fig, ax = plt.subplots(1,1, figsize = (3,3))
edge_color={
    "Apr-25":"none",
    "Apr-24":"none",
    "Aug-24":"none",
    "Jun-24":"k",
    "Nov-24":"none"
}

ax, pca = scplt.plot_pca(ax, pdata_fig2_region_norm, edge_color='Batch', color='region', cmap=cns_color, edge_cmap=edge_color, s=20, alpha=.8, return_fit=True, force=True, add_ellipses=False)


ax.set_xlabel(ax.get_xlabel(), fontsize=13, labelpad=4)
ax.set_ylabel(ax.get_ylabel(), fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

scplt.shift_legend(ax)
plt.show()

fig, ax = plt.subplots(1,1, figsize = (3,3))
edge_color={
    "Apr-25":"none",
    "Apr-24":"k",
    "Aug-24":"k",
    "Jun-24":"none",
    "Nov-24":"none"
}

ax, pca = scplt.plot_pca(ax, pdata_fig2_region_norm, edge_color='Batch', color='region', cmap=cns_color, edge_cmap=edge_color, s=20, alpha=.8, return_fit=True, force=True, add_ellipses=False)


ax.set_xlabel(ax.get_xlabel(), fontsize=13, labelpad=4)
ax.set_ylabel(ax.get_ylabel(), fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

scplt.shift_legend(ax)
plt.show()

In [ ]:
batch_to_donor = {
    "Apr-25": "batch 1",
    "Jun-24": "batch 2",
    "Apr-24": "batch 3",
    "Aug-24": "batch 3",
    "Nov-24": "batch 4",
}

df = pdata_fig2_region_norm.summary

df["donor"] = df["Batch"].map(batch_to_donor)
assert df["donor"].notna().all(), f"Unmapped batches: {df.loc[df['donor'].isna(), 'Batch'].unique()}"


pdata_fig2_region_norm.summary = df

In [ ]:
for batch in ['batch 1', 'batch 2', 'batch 3']:
    print(batch)
    pdata_batch = pdata_fig2_region_norm.filter_sample(values={'donor': batch})

    fig, ax = plt.subplots(1,1, figsize = (3,3))
    ax, pca = scplt.plot_pca(ax, pdata_batch, classes='region', s=20, alpha=.8, return_fit=True, force=True, cmap=cns_color, add_ellipses=False)
    scplt.shift_legend(ax)


    ax.set_title(batch)
    ax.set_xlabel(ax.get_xlabel(), fontsize=13, labelpad=4)
    ax.set_ylabel(ax.get_ylabel(), fontsize=13, labelpad=4)
    ax.tick_params(axis='x', labelsize=11)
    ax.tick_params(axis='y', labelsize=11)

    plt.savefig(f"MANUSCRIPT_revision/REVISION_Fig2-CNS_filtered-PCA-{batch}.svg", bbox_inches="tight", pad_inches=0.1)

#### mixed DE

In [ ]:
batch_to_donor = {
    "Apr-25": "batch 1",
    "Jun-24": "batch 2",
    "Apr-24": "batch 3",
    "Aug-24": "batch 3",
    "Nov-24": "batch 4",
}

df = pdata_fig2_region_norm.summary

df["donor"] = df["Batch"].map(batch_to_donor)
assert df["donor"].notna().all(), f"Unmapped batches: {df.loc[df['donor'].isna(), 'Batch'].unique()}"


pdata_fig2_region_norm.summary = df

In [ ]:
pdata_fig2_region_norm.summary.groupby("donor")["region"].value_counts()

In [ ]:
pdata_fig2_mixedeffect = pdata_fig2_region_norm.copy()
pdata_fig2_mixedeffect.log_transform(layer="X", base=2)

pdata_fig2_mixedeffect.show_layer_provenance()

In [ ]:
df = pdata_fig2_mixedeffect.mixed_de(
    values = [{'region': 'Cortex'}, {'region': 'SNpc'}],
    donor_col="donor",
    random_effects="intercept",
)

In [ ]:
pdata_fig2_mixedeffect.stats["mixed: Cortex vs SNpc | donor=donor"]

In [ ]:
pdata_fig2_mixedeffect.stats["mixed: Cortex vs SNpc | donor=donor"].to_csv('MANUSCRIPT_revision/REVISION_Fig2-CNS_VOLCANO-directlfq-MixedModel.csv')

In [ ]:
key = "mixed: Cortex vs SNpc | donor=donor"
color_dict={'upregulated':"#D19DCB", 'downregulated': "#85BE9E",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(3, 3))
ax, volcano_df = scplt.plot_volcano(ax, de_data=pdata_fig2_mixedeffect.stats[key], correct_fdr=True,
                   threshold=0.05, alpha=0.3, color=color_dict, label=[0,0], return_df=True,
                   group_annot_kwargs={"pos": {"group1_xy": (0.98, 1.09), "group2_xy": (0.02, 1.09)}, 'mixed_xy':(0.5,1.25)},)


cortex_markers=["Nrgn","Celf2","Camk2a","Slc17a7","Homer1","Olfm1","Phactr1","Khdrbs3","Ptprd","Cacnb4","Gria3","Icam5"]
snpc_markers = ["Th","Ddc","Ncam1","Gap43","Vat1","Scg2","Chl1",'Dpp6',"Gaa","Rps8","Vat1l","Slc6a3","Sncg","Calb2","Aldh1a1","Akr1b7"]
ax, texts = scplt.mark_volcano(ax, volcano_df, label=["Sncg", "Ddc", "Th","Slc17a7", "Camk2a","Gria3", "Homer1", "Aldh1a1", "Calb2","Akr1b7"], label_color='k', return_texts=True)
scplt.volcano_adjust_and_outline_texts(texts, expand=(3,3))

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig2-CNS_VOLCANO-directlfq-MixedModelLabeled.svg", bbox_inches="tight", pad_inches=0.1)

key = "mixed: Cortex vs SNpc | donor=donor"
color_dict={'upregulated':"#D19DCB", 'downregulated': "#85BE9E",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(3, 3))
ax, volcano_df = scplt.plot_volcano(ax, de_data=pdata_fig2_mixedeffect.stats[key], correct_fdr=True,
                   threshold=0.05, alpha=0.3, color=color_dict, label=[0,0], return_df=True,
                   group_annot_kwargs={"pos": {"group1_xy": (0.98, 1.09), "group2_xy": (0.02, 1.09)}, 'mixed_xy':(0.5,1.25)},)


cortex_markers=["Nrgn","Celf2","Camk2a","Slc17a7","Homer1","Olfm1","Phactr1","Khdrbs3","Ptprd","Cacnb4","Gria3","Icam5"]
snpc_markers = ["Th","Ddc","Ncam1","Gap43","Vat1","Scg2","Chl1",'Dpp6',"Gaa","Rps8","Vat1l","Slc6a3","Sncg","Calb2","Aldh1a1","Akr1b7"]
ax, texts = scplt.mark_volcano(ax, volcano_df, label=cortex_markers+snpc_markers, label_color='k', return_texts=True)
scplt.volcano_adjust_and_outline_texts(texts, expand=(3,3))

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig2-CNS_VOLCANO-directlfq-MixedModelLabeledSupp.svg", bbox_inches="tight", pad_inches=0.1)

## top 10

fig, ax = plt.subplots(figsize=(3, 3))
ax, volcano_df = scplt.plot_volcano(ax, de_data=pdata_fig2_mixedeffect.stats[key], correct_fdr=True,
                   threshold=0.05, alpha=0.3, color=color_dict, label=[10,10], return_df=True,
                   group_annot_kwargs={"pos": {"group1_xy": (0.98, 1.09), "group2_xy": (0.02, 1.09)}, 'mixed_xy':(0.5,1.25)},)

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig2-CNS_VOLCANO-directlfq-MixedModelTop10.svg", bbox_inches="tight", pad_inches=0.1)

In [ ]:
volcano_df.log2fc.describe()

#### adjusted p values

In [ ]:
case_values = [{'region': 'Cortex'}, {'region': 'SNpc'}]

color_dict={'upregulated':"#D19DCB", 'downregulated': "#85BE9E",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(3, 3))
ax, volcano_df = scplt.plot_volcano(ax, pdata_fig2_region_norm, values=case_values, threshold=0.05, return_df=True, alpha=0.3,
                                    fold_change_mode='mean', color=color_dict, label=[0,0], correct_fdr=True,
                                    group_annot_kwargs={"pos": {"group1_xy": (0.98, 1.09), "group2_xy": (0.02, 1.09)}},)

ax, texts = scplt.mark_volcano(ax, volcano_df, label=["Sncg", "Ddc", "Th","Slc17a7", "Camk2a","Gria3", "Homer1", "Aldh1a1", "Calb2","Akr1b7"], label_color='k', return_texts=True)

scplt.volcano_adjust_and_outline_texts(texts, expand=(3,3))


ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.show()
volcano_df.to_csv("MANUSCRIPT_revision/REVISION_Fig2-CNS_DE-directlfq-adjP.csv")

In [ ]:
case_values = [{'region': 'Cortex'}, {'region': 'SNpc'}]

color_dict={'upregulated':"#D19DCB", 'downregulated': "#85BE9E",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(3, 3))
ax, volcano_df = scplt.plot_volcano(ax, pdata_fig2_region_norm_impute,
                                    values=case_values,
                                    threshold=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', color=color_dict, label=[0,0],
                                    group_annot_kwargs={"pos": {"group1_xy": (0.98, 1.09), "group2_xy": (0.02, 1.09)}},)


ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

volcano_df.to_csv("MANUSCRIPT_revision/REVISION_Fig2-CNS_DE-impute-adjP.csv")

# Fig 4 - non-neuronal cells

## astrocyte vs neuron

### pre-processing

In [ ]:
# take subset of data from pdata_all_region_filtered

pdata_fig4_region_filtered = pdata_all_region_filtered.filter_prot_significant()
pdata_fig4_region_filtered = pdata_fig4_region_filtered.filter_sample(condition="Grouping != 'snpc'")
pdata_fig4_region_filtered = pdata_fig4_region_filtered.filter_sample(condition="Batch != 'Apr-25'")

mapping = {
    "ctx": "Neuron",
    "ast": "Astrocyte",
}
pdata_fig4_region_filtered.summary["cell_type"] = pdata_fig4_region_filtered.summary["Grouping"].replace(mapping)
pdata_fig4_region_filtered.update_summary()

In [ ]:
pdata_fig4_region_processed = pdata_fig4_region_filtered.copy()
pdata_fig4_region_processed = pdata_fig4_region_processed.filter_sample(min_prot=1800)

pdata_fig4_region_processed = pdata_fig4_region_processed.filter_prot_found(min_ratio=0.4, group = ['cell_type'], match_any=True)
pdata_fig4_region_processed = pdata_fig4_region_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig4_region_norm = pdata_fig4_region_processed.copy()
pdata_fig4_region_norm.normalize(method='directlfq')

print(pdata_fig4_region_norm.summary['Grouping'].value_counts())
print(pdata_fig4_region_norm.summary['Batch'].value_counts())
print(pdata_fig4_region_norm.summary['cell_type'].value_counts())

### plots

In [ ]:
cell_type_color={
    "Neuron": "#D19DCB",
    "Astrocyte": "#84C8D5" # "#CAEEFB"
}

In [ ]:
case1 = {'cell_type': 'Neuron'}
case2 = {'cell_type': 'Astrocyte'}
case_values = [case1, case2]

cell_type_color={
    "Neuron": "#D19DCB",
    "Astrocyte": "#84C8D5" # "#CAEEFB"
}
color_dict={'upregulated': cell_type_color['Neuron'], 'downregulated': cell_type_color['Astrocyte'],'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(3.63, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_fig4_region_norm,
                                    values=case_values,
                                    threshold=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', color=color_dict, label=[10,10])

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig4-non-neuron_VOLCANO-directlfq-top10-adjP.svg", bbox_inches="tight", pad_inches=0.1)
volcano_df.to_csv("MANUSCRIPT_revision/REVISION_Fig4-non-neuron_DE-directlfq-adjP.csv")

## SWI model

### pre-processing

In [ ]:
obs_columns = ['date', 'acquisition', 'size', 'cell_type', 'replicate']

pdata_fig4_swi = pAnnData.import_data(source_type='diann', report_file = 'data/2511_ast-SWI-only/report.parquet', obs_columns=obs_columns)

In [ ]:
pdata_fig4_swi_filtered = pdata_fig4_swi.filter_sample(condition="cell_type not in ['T2','TEAB35']")
pdata_fig4_swi_filtered = pdata_fig4_swi_filtered.filter_prot_significant()
pdata_fig4_swi_filtered = pdata_fig4_swi_filtered.filter_sample(min_prot=1000)

mapping = {
    "scartissue": "SWI",
    "astrocyte": "Uninjured",
}
pdata_fig4_swi_filtered.summary["condition"] = pdata_fig4_swi_filtered.summary["cell_type"].replace(mapping)
pdata_fig4_swi_filtered.update_summary()

pdata_fig4_swi_filtered.prot.obs.condition.value_counts()

In [ ]:
pdata_fig4_swi_processed = pdata_fig4_swi_filtered.copy()
pdata_fig4_swi_processed = pdata_fig4_swi_processed.filter_prot_found(min_ratio=0.4, group = ['condition'], match_any=True)
pdata_fig4_swi_processed = pdata_fig4_swi_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig4_swi_norm = pdata_fig4_swi_processed.copy()
pdata_fig4_swi_norm.normalize(method='directlfq')

### plots

In [ ]:
condition_color={
    "SWI": "#1F98B1",
    "Uninjured": "#ACDFE9" # "#CAEEFB"
}

In [ ]:
case_values = [{'condition': 'SWI'}, {'condition': 'Uninjured'}]

color_dict={'upregulated':condition_color['SWI'], 'downregulated': condition_color['Uninjured'],'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(3.34, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_fig4_swi_norm, values = case_values, threshold = 0.05, return_df=True, correct_fdr=True, color=color_dict, label=[0,0])
scplt.mark_volcano(ax, volcano_df, label=['Aldh1l1','Gfap'], label_color='k')

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

volcano_df.to_csv("MANUSCRIPT_revision/REVISION_Fig4-SWI_DE-directlfq-adjP.csv")

# Fig 5 - snpc subtypes

## anxa

### pre-processing

In [ ]:
pdata_fig3_snpc_filter = pdata_all_region_filtered.filter_sample(condition ='Grouping == "snpc"')
pdata_fig3_snpc_filter = pdata_fig3_snpc_filter.filter_sample(condition = 'Batch == "Apr-25"')
pdata_fig3_snpc_filter = pdata_fig3_snpc_filter.filter_prot_significant()
pdata_fig3_snpc_filter = pdata_fig3_snpc_filter.filter_sample(min_prot=1000)

pdata_fig3_snpc_filter.summary["cell_type"] = pdata_fig3_snpc_filter.summary["Sub grouping"].replace({
    "anxaY": "Anxa+",
    "anxaN": "Anxa-"
})
pdata_fig3_snpc_filter.update_summary()

pdata_fig3_snpc_processed = pdata_fig3_snpc_filter.copy()
pdata_fig3_snpc_processed = pdata_fig3_snpc_processed.filter_prot_found(min_ratio=0.4, group = ['Sub grouping'], match_any=True)
pdata_fig3_snpc_processed = pdata_fig3_snpc_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig3_snpc_norm = pdata_fig3_snpc_processed.copy()
pdata_fig3_snpc_norm.normalize(method='directlfq')

In [ ]:
pd.DataFrame(pdata_fig3_snpc_norm.summary)

### plots

In [ ]:
anxa_color={
    "Anxa-": "#A0FFA0",
    "Anxa+": "white"
}

In [ ]:
case_values = [{'cell_type': 'Anxa+'}, {'cell_type': 'Anxa-'}]

color_dict={'upregulated':anxa_color['Anxa+'], 'downregulated': anxa_color['Anxa-'],'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_fig3_snpc_norm,
                                    values=case_values,
                                    threshold=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', color=color_dict, label=[5,5])

white = np.array([1.0, 1.0, 1.0])
green = np.array([160/255, 1.0, 160/255])  # #A0FFA0

for coll in ax.collections:
    fc = coll.get_facecolors()
    if fc is None or fc.size == 0:
        continue

    mask_white = (fc[:, :3] == white).all(axis=1)
    mask_green = (fc[:, :3] == green).all(axis=1)
    mask = mask_white | mask_green

    if mask.any():
        ec = np.zeros_like(fc)          # transparent edges by default
        ec[mask] = [0, 0, 0, 1]         # black outline
        coll.set_edgecolors(ec)

        lw = np.zeros(len(fc))
        lw[mask] = 0.4
        coll.set_linewidths(lw)

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

volcano_df.to_csv("MANUSCRIPT_revision/REVISION_Fig3-Anxa_DE-directlfq-adjP.csv")

# Fig 6 - 6mo mouse (snpc)

### pre-processing

In [ ]:
pdata_6mo_diann_base = pAnnData.import_data(source_type='diann', report_file = 'data/2511_Mus6moSnpcONLY/report.parquet')

In [ ]:
colnames = [
    'date',
    'gradient',
    'sample_id',
    'size',
    'confirmation',
    'thickness',
    'sample',
    'organism',
    'region',
    'well_position'
]

# ignore 11-token parsing type on both
pdata_6mo_diann = pdata_6mo_diann_base.filter_sample(condition='parsingType == "10-tokens"')
pdata_6mo_diann.summary = scutils.parse_filename_index(pdata_6mo_diann.summary, colnames)

pdata_6mo_processed = pdata_6mo_diann.filter_prot_significant()

In [ ]:
pdata_6mo_snpc = pdata_6mo_processed.filter_sample(condition='region == "snpc"')
pdata_6mo_snpc = pdata_6mo_snpc.filter_sample(min_prot=1000)
pdata_6mo_snpc.summary["sample"] = pdata_6mo_snpc.summary["sample"].replace({
    "6mo-aggY": "Agg+",
    "6mo-aggN": "Agg-"
})
pdata_6mo_snpc.update_summary()

pdata_6mo_snpc_filter = pdata_6mo_snpc.copy()
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_prot_found(min_ratio=0.4, group=['sample'],match_any=True)
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_prot(valid_genes=True, unique_profiles=True)
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_sample(min_prot=1000)

pdata_6mo_snpc_norm = pdata_6mo_snpc_filter.copy()
pdata_6mo_snpc_norm.normalize(method='directlfq')

pdata_6mo_snpc_norm.summary['sample'].value_counts()

In [ ]:
pdata_6mo_snpc_norm_RF = pdata_6mo_snpc_filter.copy()
pdata_6mo_snpc_norm_RF.normalize(method="reference_feature",
    reference_columns=["Tubb5", "Gapdh"]
)

### plots

In [ ]:
agg_color={
    "Agg+": "#C64D4A",
    "Agg-": "#BFBFBF"
}

In [ ]:
# PEPTIDE ABUNDANCE DIST

import matplotlib.patches as mpatches
from scipy import stats

agg_color = {"Agg+": "#C64D4A", "Agg-": "#BFBFBF"}
agg_order = ["Agg+", "Agg-"]

df_raw = pdata_6mo_snpc_filter.get_abundance(on="peptide", classes="sample", log=False)
df_raw = df_raw[df_raw["abundance"] > 0].copy()
df_raw["log10_abundance"] = np.log10(df_raw["abundance"])

sample_meta = df_raw.groupby("cell")["class"].first()
sample_order = (
    sample_meta[sample_meta == "Agg+"].index.tolist()
    + sample_meta[sample_meta == "Agg-"].index.tolist()
)
short_labels = [s.split("_")[-1] for s in sample_order]
violin_colors = [agg_color[sample_meta[s]] for s in sample_order]

fig, ax = plt.subplots(figsize=(14, 4))

sns.violinplot(
    data=df_raw,
    x="cell",
    y="log10_abundance",
    order=sample_order,
    palette=violin_colors,
    inner="box",
    cut=0,
    linewidth=1,
    bw_adjust=1.5,
    gridsize=200,
    ax=ax,
)

xs = [sample_order.index(c) for c in df_raw["cell"]]
ax.scatter(xs, df_raw["log10_abundance"], color="black", s=1.3**2, alpha=0.12, linewidths=0, zorder=3)

ax.set_xlabel("Sample")
ax.set_ylabel(r"$\log_{10}$(Peptide abundance)")
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(short_labels, rotation=90)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

handles = [mpatches.Patch(color=agg_color[k], label=k) for k in agg_order]
ax.legend(handles=handles, title="Agg status", frameon=True,
          bbox_to_anchor=(0.02, 1.2), loc="upper left", borderaxespad=0)

plt.tight_layout()
plt.savefig("MANUSCRIPT_revision/REVISION_Fig6-Abundance_Peptide_Violin.svg", bbox_inches="tight", pad_inches=0.1)

# t-test on per-sample medians
medians = df_raw.groupby("cell")["log10_abundance"].median()
agg_plus  = medians[sample_meta[medians.index] == "Agg+"]
agg_minus = medians[sample_meta[medians.index] == "Agg-"]
t, p = stats.ttest_ind(agg_plus, agg_minus)
print(f"Peptide abundance: Agg+ median={agg_plus.mean():.3f} (n={len(agg_plus)}), "
      f"Agg- median={agg_minus.mean():.3f} (n={len(agg_minus)}), t={t:.3f}, p={p:.4f}")

In [ ]:
# PROTEIN ABUNDANCE DIST (directLFQ)

import matplotlib.patches as mpatches

agg_color = {"Agg+": "#C64D4A", "Agg-": "#BFBFBF"}
agg_color_raw = {"Agg+": "#E8A8A7", "Agg-": "#E0E0E0"}  # lighter versions for raw

agg_order = ["Agg+", "Agg-"]

# --- load both datasets and tag them ---
df_raw = pdata_6mo_snpc_filter.get_abundance(on="protein", classes="sample", log=False)
df_raw = df_raw[df_raw["abundance"] > 0].copy()
df_raw["log10_abundance"] = np.log10(df_raw["abundance"])
df_raw["norm_status"] = "Raw"

df_norm = pdata_6mo_snpc_norm.get_abundance(on="protein", classes="sample", log=False)
df_norm = df_norm[df_norm["abundance"] > 0].copy()
df_norm["log10_abundance"] = np.log10(df_norm["abundance"])
df_norm["norm_status"] = "Normalized"

df = pd.concat([df_raw, df_norm], ignore_index=True)

sample_meta = df_raw.groupby("cell")["class"].first()
sample_order = (
    sample_meta[sample_meta == "Agg+"].index.tolist()
    + sample_meta[sample_meta == "Agg-"].index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 4))

sns.violinplot(
    data=df,
    x="cell",
    y="log10_abundance",
    hue="norm_status",
    hue_order=["Raw", "Normalized"],
    order=sample_order,
    split=True,
    inner="box",
    cut=0,
    linewidth=1,
    bw_adjust=1.5,
    gridsize=200,
    ax=ax,
)

# re-color violin halves: left=Raw (light), right=Normalized (dark)
for i, sample in enumerate(sample_order):
    agg = sample_meta[sample]
    ax.collections[i * 2].set_facecolor(agg_color_raw[agg])      # left half = raw
    ax.collections[i * 2 + 1].set_facecolor(agg_color[agg])      # right half = normalized

# stripplot offsets
for norm_status, offset in [("Raw", -0.15), ("Normalized", 0.15)]:
    sub = df[df["norm_status"] == norm_status]
    xs = [sample_order.index(c) + offset for c in sub["cell"]]
    ax.scatter(xs, sub["log10_abundance"], color="black", s=1.3**2, alpha=0.12, linewidths=0, zorder=3)

short_labels = [s.split("_")[-1] for s in sample_order]
ax.set_xlabel("Sample")
ax.set_ylabel(r"$\log_{10}$(Protein abundance)")
ax.set_xticks(range(len(sample_order)))
ax.set_xticklabels(short_labels, rotation=90)
ax.tick_params(axis="x", rotation=90)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# legend: one row per Agg status, showing light (raw) and dark (norm) patches
legend_handles = []
for agg in agg_order:
    legend_handles += [
        mpatches.Patch(color=agg_color_raw[agg], label=f"{agg} - Raw"),
        mpatches.Patch(color=agg_color[agg],     label=f"{agg} - Normalized"),
    ]

ax.legend(
    handles=legend_handles,
    title="Sample / Normalization",
    frameon=True,
    ncol=2,
    bbox_to_anchor=(0.02, 1.2),
    loc="upper left",
    borderaxespad=0,
)

plt.tight_layout()
plt.savefig("MANUSCRIPT_revision/REVISION_Fig6-Abundance_SplitViolin.svg", bbox_inches="tight", pad_inches=0.1)

from scipy import stats

for label, data in [("Raw (pre-normalization)", df_raw), ("Normalized", df_norm)]:
    medians = data.groupby("cell")["log10_abundance"].median()
    agg_plus  = medians[sample_meta[medians.index] == "Agg+"]
    agg_minus = medians[sample_meta[medians.index] == "Agg-"]
    t, p = stats.ttest_ind(agg_plus, agg_minus)
    print(f"{label}: Agg+ median={agg_plus.mean():.3f} (n={len(agg_plus)} cells), Agg- median={agg_minus.mean():.3f} (n={len(agg_minus)} cells), t={t:.3f}, p={p:.4f}")

In [ ]:
pdata_6mo_snpc_norm.prot.var

In [ ]:
case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True,
                                    fold_change_mode='mean', color=color_dict, label=[0,0])

scplt.mark_volcano(ax, volcano_df, label=['Gapdh','Tubb5','Actb;Actg1','Pgk1'], label_color="#030304")

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_VOLCANO-directlfq-Housekeeping.svg", bbox_inches="tight", pad_inches=0.1)

In [ ]:
volcano_df

In [ ]:
pdata_6mo_snpc_norm.export_layer("X", f"MANUSCRIPT_revision/REVISION_Fig6-mus6moSNpc_normalized.csv", transpose=True, var_names='Genes', obs_names='sample')

In [ ]:
vol_list = volcano_df[volcano_df['significance'].isin(["upregulated", "downregulated","not comparable"])].Genes.to_list()

mus6mosnpc_sample = pd.read_csv("MANUSCRIPT_revision/REVISION_Fig6-mus6moSNpc_normalized.csv")
mus6mosnpc_sample_filtered = mus6mosnpc_sample[mus6mosnpc_sample['Genes'].isin(vol_list)].copy()

values=['Agg+','Agg-']
mus6mosnpc_sample_filtered = mus6mosnpc_sample_filtered.filter(regex='Agg+|Agg-|^Genes$')

# map significance from df_YHC onto the filtered df via Genes
sig_map = volcano_df.set_index('Genes')['significance']
mus6mosnpc_sample_filtered['significance'] = mus6mosnpc_sample_filtered['Genes'].map(sig_map)

# sort columns, keeping the first column first and significance last
first_col = mus6mosnpc_sample_filtered.columns[0]
other_cols = sorted(c for c in mus6mosnpc_sample_filtered.columns if c not in [first_col, 'significance'])
mus6mosnpc_sample_filtered = mus6mosnpc_sample_filtered[[first_col] + other_cols + ['significance']]

mus6mosnpc_sample_filtered.to_csv("MANUSCRIPT_revision/REVISION_Fig6-mus6moSNpc_volcano-only.csv", index=False)

In [ ]:
case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True,
                                    fold_change_mode='mean', color=color_dict, label=[0,0])

scplt.mark_volcano(ax, volcano_df, label=['Gapdh','Tubb5','Actb;Actg1','Pgk1','Sqstm1'], label_color="#030304")

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

In [ ]:

scplt.plot_abundance_boxgrid(pdata=pdata_6mo_snpc_norm, namelist=['Gapdh','Tubb5'], classes='sample', palette=agg_color, sig_pairs=True, figsize=(2,2.5), sig_kwargs={"fontsize":8}, log_scale=True)
plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_ABUNDANCE-GapdhTubb5.svg", bbox_inches="tight", pad_inches=0.1)

scplt.plot_abundance_boxgrid(pdata=pdata_6mo_snpc_norm, namelist=['Actb;Actg1','Pgk1'], classes='sample', palette=agg_color, sig_pairs=True, figsize=(2,2.5), sig_kwargs={"fontsize":8}, log_scale=True)
plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_ABUNDANCE-ActbPgk1.svg", bbox_inches="tight", pad_inches=0.1)

In [ ]:
# with ADJUSTED P VALUE
 
case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', color=color_dict, label=[0,0])

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_VOLCANO-adjP.svg", bbox_inches="tight", pad_inches=0.1)
volcano_df.to_csv("MANUSCRIPT_revision/MANUSCRIPT_Fig5_DE-directlfq-adjP.csv")

In [ ]:
def extract_gene_group(volcano_df, prefixes, top_n=5):
    """
    prefixes: list of lowercase prefixes, e.g. ['atp5']
    """
    mask_prefix = volcano_df["Genes"].str.lower().apply(
        lambda g: any(g.startswith(p) for p in prefixes)
    )
    sig_mask = (volcano_df["significance"] == 'upregulated') | \
               (volcano_df["significance"] == 'downregulated')

    gene_list = volcano_df.loc[mask_prefix & sig_mask, "Genes"].unique().tolist()

    if len(gene_list) == 0:
        return [], [], []

    top = (
        volcano_df[volcano_df["Genes"].isin(gene_list)]
        .sort_values("significance_score", ascending=True)
        .head(top_n)["Genes"]
        .tolist()
    )
    others = [g for g in gene_list if g not in top]

    return gene_list, top, others

In [ ]:
## RPL/RPS
rpl_list, rpl_top5, rpl_others = extract_gene_group(volcano_df, prefixes=["rpl"])
rps_list, rps_top5, rps_others = extract_gene_group(volcano_df, prefixes=["rps"])

print("rpl_list: ", len(rpl_list))
print("rps_list: ", len(rps_list))

case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}
rps_dict={'downregulated': '#5166FF'}
rpl_dict={'downregulated': '#1F2CCF'}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    threshold=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', no_marks=True)

texts = []
ax, t = scplt.mark_volcano(ax, volcano_df, label=rpl_top5, label_color='#1F2CCF',return_texts=True)
texts.extend(t)
ax, t = scplt.mark_volcano_by_significance(ax, volcano_df, label=rps_top5, color=rps_dict,return_texts=True)
texts.extend(t)
scplt.mark_volcano_by_significance(ax, volcano_df, label=rpl_others, color=rpl_dict, show_names=False)
scplt.mark_volcano_by_significance(ax, volcano_df, label=rps_others, color=rps_dict, show_names=False)

scplt.volcano_adjust_and_outline_texts(texts, expand=(2,2))

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_VOLCANO-directlfq-RplRps.svg", bbox_inches="tight", pad_inches=0.1)

In [ ]:
## Proteasome
proteasome_list, proteasome_top5, proteasome_others = extract_gene_group(volcano_df, prefixes=["psm"])
proteasome_list.append('Uchl1')

print("proteasome_list: ", len(proteasome_list))

case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    threshold=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', no_marks=True)

texts = []
ax, t = scplt.mark_volcano_by_significance(ax, volcano_df, label=proteasome_list, color=color_dict, return_texts=True)
texts.extend(t)
scplt.volcano_adjust_and_outline_texts(texts, expand=(2, 2))

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_VOLCANO-directlfq-Proteasome.svg", bbox_inches="tight", pad_inches=0.1)

In [ ]:
## Mitochondria

mito_groups = {
    "complex5":   ["atp5"],
    "complex4":   ["cox"],
    "complex1":   ["nduf"],
    "complex2":   ["sdh"],
    "complex3":   ["uqcr"],
    "mito_other": ["cyc1", "cycs", "pgk", "vdac"],
}

mito_results = {}

for group_name, prefixes in mito_groups.items():
    gene_list, top5, others = extract_gene_group(volcano_df, prefixes)
    mito_results[group_name] = {
        "list": gene_list,
        "top5": top5,
        "others": others,
    }

mito_color_dict = {
    "complex1": {     # NDUF*
        "upregulated":   "#D64545",   # strong red
        "downregulated": "#1E3A8A",   # deep indigo blue
    },
    "complex2": {     # SDH*
        "upregulated":   "#C03636",   # darker red
        "downregulated": "#1F4DA0",   # strong royal blue
    },
    "complex3": {     # UQCR*
        "upregulated":   "#B42E2E",   # deep cherry
        "downregulated": "#2665C2",   # vivid blue
    },
    "complex4": {     # COX*
        "upregulated":   "#A82828",   # wine red
        "downregulated": "#2D7AE6",   # bright azure
    },
    "complex5": {     # ATP5*
        "upregulated":   "#9E2020",   # maroon
        "downregulated": "#3D8BFF",   # clear blue
    },
    "mito_other": {
        "upregulated":   "#8C1A1A",   # dark brick red
        "downregulated": "#4C9CFF",   # lighter but still strong blue
    },
}

## Mitochondria, all together

case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    threshold=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', no_marks=True)

texts = []
for group_name, d in mito_results.items():
    colors = mito_color_dict[group_name]

    if len(d["top5"]) > 0:
        # highlight top 5 (with names)
        ax, t = scplt.mark_volcano_by_significance(
            ax, volcano_df,
            label=d["top5"],
            color=colors,
            return_texts=True,
        )

        texts.extend(t)

    if len(d["others"]) > 0:
        # highlight the others (no names)
        scplt.mark_volcano_by_significance(
            ax, volcano_df,
            label=d["others"],
            color=colors,
            show_names=False,
        )

scplt.volcano_adjust_and_outline_texts(texts, expand=(2,2))

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_Fig5_VOLCANO-directlfq-MitoAll.svg", bbox_inches="tight", pad_inches=0.1)

### heatmap

In [ ]:
case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_6mo_snpc_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True,
                                    fold_change_mode='mean', color=color_dict, label=[0,0])

scplt.mark_volcano(ax, volcano_df, label=['Gapdh','Tubb5','Actb;Actg1','Pgk1','Sqstm1'], label_color="#030304")

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

In [ ]:
def extract_gene_group(volcano_df, prefixes, top_n=5):
    """
    prefixes: list of lowercase prefixes, e.g. ['atp5']
    """
    mask_prefix = volcano_df["Genes"].str.lower().apply(
        lambda g: any(g.startswith(p) for p in prefixes)
    )
    sig_mask = (volcano_df["significance"] == 'upregulated') | \
               (volcano_df["significance"] == 'downregulated')

    gene_list = volcano_df.loc[mask_prefix & sig_mask, "Genes"].unique().tolist()

    if len(gene_list) == 0:
        return [], [], []

    top = (
        volcano_df[volcano_df["Genes"].isin(gene_list)]
        .sort_values("significance_score", ascending=True)
        .head(top_n)["Genes"]
        .tolist()
    )
    others = [g for g in gene_list if g not in top]

    return gene_list, top, others

rpl_list, rpl_top5, rpl_others = extract_gene_group(volcano_df, prefixes=["rpl"])
rps_list, rps_top5, rps_others = extract_gene_group(volcano_df, prefixes=["rps"])

proteasome_list, proteasome_top5, proteasome_others = extract_gene_group(volcano_df, prefixes=["psm"])
proteasome_list.append('Uchl1')

mito_groups = {
    "complex5":   ["atp5"],
    "complex4":   ["cox"],
    "complex1":   ["nduf"],
    "complex2":   ["sdh"],
    "complex3":   ["uqcr"],
    "mito_other": ["cyc1", "cycs", "pgk", "vdac"],
}

mito_results = {}
for group_name, prefixes in mito_groups.items():
    gene_list, top5, others = extract_gene_group(volcano_df, prefixes)
    mito_results[group_name] = {
        "list": gene_list,
        "top5": top5,
        "others": others,
    }

print(len(rpl_list))
print(len(rps_list))
print(len(proteasome_list))
for group_name, d in mito_results.items():
    print(len(d['list']))


protein_groups = {
    "Ribosome": rpl_list + rps_list, 
    "Proteasome": proteasome_list,
    "Mitochondria": [gene for d in mito_results.values() for gene in d["list"]]
    }

In [ ]:
agg_color={
    "Agg+": "#C64D4A",
    "Agg-": "#BFBFBF"
}

fig, fig_legend = pdata_6mo_snpc_norm.plot_clustered_heatmap(
    classes=["sample"],
    stats_key="Agg+ vs Agg-",
    significance_categories=["upregulated", "downregulated"],  # default
    sort_by={"sample": ["Agg-", "Agg+"]},
    # optional annotation overlay:
    protein_groups=protein_groups,
    group_colors={"Ribosome": "#6EDC00", "Proteasome": "#F4D03F", "Mitochondria": "#FF0000",},
    header_colors=agg_color,
    show_unassigned=False,
    sample_label_col="sample_id",
    label_color="Black",
    separate_legend=True,
    figsize=(3,10),
    dendrogram_linewidth=0.3
)

fig.savefig("MANUSCRIPT_revision/REVISION_Fig5_Heatmap.svg", bbox_inches="tight", pad_inches=0.1)
fig_legend.savefig("MANUSCRIPT_revision/REVISION_Fig5_Heatmap_legend.svg", bbox_inches="tight", pad_inches=0.1)

In [ ]:
fig.axes

# Fig X - mouse (1mo/3mo/6mo cortex + snpc)

## pre-processing

In [ ]:
pdata_mus1mo3mo6mo_diann_snpc_base = pAnnData.import_data(source_type='diann', report_file = 'data/2511_MusSNPC-1mo-3mo-6mo-combined/report.parquet')

In [ ]:
colnames = [
    'date',
    'gradient',
    'sample_id',
    'size',
    'confirmation',
    'thickness',
    'sample',
    'organism',
    'region',
    'well_position'
]

#### SNPC

pdata_mus1mo3mo6mo_diann_snpc = pdata_mus1mo3mo6mo_diann_snpc_base.filter_sample(condition='parsingType == "10-tokens"')
pdata_mus1mo3mo6mo_diann_snpc.summary = scutils.parse_filename_index(pdata_mus1mo3mo6mo_diann_snpc.summary, colnames)

pdata_mus1mo3mo6mo_processed_snpc = pdata_mus1mo3mo6mo_diann_snpc.filter_prot_significant()

pdata_mus1mo3mo6mo_processed_snpc = pdata_mus1mo3mo6mo_processed_snpc.filter_sample(condition='region == "snpc"')
pdata_mus1mo3mo6mo_processed_snpc.summary[["timepoint", "sample"]] = pdata_mus1mo3mo6mo_processed_snpc.summary["sample"].str.extract(r"(\d+mo)-(agg[YN])")
pdata_mus1mo3mo6mo_processed_snpc.summary["sample"] = pdata_mus1mo3mo6mo_processed_snpc.summary["sample"].map({"aggY": "Agg+", "aggN": "Agg-"})
pdata_mus1mo3mo6mo_processed_snpc.update_summary()
pdata_mus1mo3mo6mo_processed_snpc = pdata_mus1mo3mo6mo_processed_snpc.filter_sample(min_prot=1000)

In [ ]:
pd.DataFrame(pdata_mus1mo3mo6mo_processed_snpc.summary.groupby(['timepoint','sample']).count())

In [ ]:
pdata_3mo_snpc_filter = pdata_mus1mo3mo6mo_processed_snpc.filter_sample(condition='timepoint == "3mo"')
pdata_3mo_snpc_filter = pdata_3mo_snpc_filter.filter_prot_found(min_ratio=0.4, group=['sample'],match_any=True)
pdata_3mo_snpc_filter = pdata_3mo_snpc_filter.filter_prot(valid_genes=True, unique_profiles=True)
pdata_3mo_snpc_filter = pdata_3mo_snpc_filter.filter_sample(min_prot=1000)

pdata_3mo_snpc_norm = pdata_3mo_snpc_filter.copy()
pdata_3mo_snpc_norm.normalize(method='directlfq')

pdata_3mo_snpc_norm.summary['sample'].value_counts()

In [ ]:
pdata_6mo_snpc_filter = pdata_mus1mo3mo6mo_processed_snpc.filter_sample(condition='timepoint == "6mo"')
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_prot_found(min_ratio=0.4, group=['sample'],match_any=True)
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_prot(valid_genes=True, unique_profiles=True)
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_sample(min_prot=1000)

pdata_6mo_snpc_norm = pdata_6mo_snpc_filter.copy()
pdata_6mo_snpc_norm.normalize(method='directlfq')

pdata_6mo_snpc_norm.summary['sample'].value_counts()

### plots

In [ ]:
agg_color={
    "Agg+": "#C64D4A",
    "Agg-": "#BFBFBF"
}

### abundance exploration

In [ ]:
plot_cortex=['RPL7L1', 'RPL37A', 'PSMD10', 'PSMB8', 'CHMP2A', 'CDK1', 'CDK2', 'HSPB8', 'MARCKS', 'RPS27L', 'MARCKSL1', 'TGM2', 'PSMB2', 'PSMB5', 'CSNK1A1', 'CSNK2B']
plot_snpc=['RPL23A', 'RPL6', 'PSMD10', 'PSMB8', 'DDC', 'LAMP2', 'PSMB2', 'PSMB5', 'RPL23']

def to_camel_case(gene):
    return gene[0].upper() + gene[1:].lower()

plot_cortex = [to_camel_case(n) for n in plot_cortex]
plot_snpc = [to_camel_case(n) for n in plot_snpc]

pdata_plot_3mo_snpc = pdata_mus1mo3mo6mo_processed_snpc.filter_sample(condition='timepoint == "3mo"')
pdata_plot_6mo_snpc = pdata_mus1mo3mo6mo_processed_snpc.filter_sample(condition='timepoint == "6mo"')

plot_names={0:"3mo snpc", 1: "6mo snpc", 2: "3mo cortex", 3: "6mo cortex"}

for i, pdata in enumerate([pdata_plot_3mo_snpc,pdata_plot_6mo_snpc]):
    print(i)
    print(plot_names[i])
    scplt.plot_abundance_boxgrid(pdata=pdata, namelist=plot_snpc, classes='sample', palette=agg_color, sig_pairs=True, figsize=(2,2.5), sig_kwargs={"fontsize":8}, log_scale=True, show_n=True)

### 3mo snpc

In [ ]:
case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_3mo_snpc_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True,
                                    fold_change_mode='mean', color=color_dict, label=[10,10])

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_FigX3moSnpc_VOLCANO-directlfq.svg", bbox_inches="tight", pad_inches=0.1)
volcano_df.to_csv("MANUSCRIPT_revision/REVISION_FigX3moSnpc-DE-directlfq.csv")

In [ ]:
case_values = [{'sample': 'Agg+'}, {'sample': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_3mo_snpc_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True, correct_fdr=True,
                                    fold_change_mode='mean', color=color_dict, label=[10,10])

ax.set_ylabel('$log_{10}$ adjusted p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

volcano_df.to_csv("MANUSCRIPT_revision/REVISION_FigX3moSnpc-DE-directlfq-adjP.csv")

In [ ]:
pdata_3mo_snpc_norm.export_layer("X", f"MANUSCRIPT_revision/REVISION_FigX-3moSnpc_normalized.csv", transpose=True, var_names='Genes', obs_names='sample')

In [ ]:
vol_list = volcano_df[volcano_df['significance'].isin(["upregulated", "downregulated"])].Genes.to_list()

m3_sample = pd.read_csv("MANUSCRIPT_revision/REVISION_FigX-3moSnpc_normalized.csv")
m3_sample_filtered = m3_sample[m3_sample['Genes'].isin(vol_list)].copy()

# map significance from volcano_df onto the filtered df via Genes
sig_map = volcano_df.set_index('Genes')['significance']
m3_sample_filtered['significance'] = m3_sample_filtered['Genes'].map(sig_map)

# sort columns, keeping the first column first and significance last
first_col = m3_sample_filtered.columns[0]
other_cols = sorted(c for c in m3_sample_filtered.columns if c not in [first_col, 'significance'])
m3_sample_filtered = m3_sample_filtered[[first_col] + other_cols + ['significance']]

m3_sample_filtered.to_csv("MANUSCRIPT_revision/REVISION_FigX-3moSnpc_volcano-only.csv", index=False)

# power analysis

### dataset prep

In [ ]:
import os

pdata_all_region = pAnnData.import_data(source_type='diann',report_file = 'data/2505_rna_prot_full/report.tsv')

# data clean-up | uses file_annotation.csv to help with annotations

file_annotation = pd.read_csv('abc_analysis/file_annotation.csv')

file_annotation["parsed_filename"] = (
    file_annotation["RAW FILE"]
    .apply(os.path.basename)
    .str.replace(".raw", "", regex=False)
)

summary = pdata_all_region.summary

summary_files = set(summary.index)
annotation_files = set(file_annotation["parsed_filename"])
missing_in_summary = annotation_files - summary_files
missing_in_annotation = summary_files - annotation_files
pdata_all_region_filtered = pdata_all_region.filter_sample(exclude_file_list=list(missing_in_annotation))

fa_sub = file_annotation[["parsed_filename", "File Name", "Grouping", "Sub grouping", "Region", "Batch"]]
fa_sub = fa_sub.set_index("parsed_filename")
fa_aligned = fa_sub.reindex(pdata_all_region_filtered.summary.index)
cols_to_add = ["File Name", "Grouping", "Sub grouping", "Region", "Batch"]
pdata_all_region_filtered.summary[cols_to_add] = fa_aligned[cols_to_add]

pdata_all_region_filtered.update_summary()

pd.DataFrame(pdata_all_region_filtered.summary)

In [ ]:
pdata_6mo_diann_base = pAnnData.import_data(source_type='diann', report_file = 'data/2511_Mus6moSnpcONLY/report.parquet')

colnames = [
    'date',
    'gradient',
    'sample_id',
    'size',
    'confirmation',
    'thickness',
    'sample',
    'organism',
    'region',
    'well_position'
]

# ignore 11-token parsing type on both
pdata_6mo_diann = pdata_6mo_diann_base.filter_sample(condition='parsingType == "10-tokens"')
pdata_6mo_diann.summary = scutils.parse_filename_index(pdata_6mo_diann.summary, colnames)

pdata_6mo_processed = pdata_6mo_diann.filter_prot_significant()

pdata_6mo_snpc = pdata_6mo_processed.filter_sample(condition='region == "snpc"')
pdata_6mo_snpc = pdata_6mo_snpc.filter_sample(min_prot=1000)
pdata_6mo_snpc.summary["sample"] = pdata_6mo_snpc.summary["sample"].replace({
    "6mo-aggY": "Agg+",
    "6mo-aggN": "Agg-"
})
pdata_6mo_snpc.update_summary()

pdata_6mo_snpc_filter = pdata_6mo_snpc.copy()
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_prot_found(min_ratio=0.4, group=['sample'],match_any=True)
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_prot(valid_genes=True, unique_profiles=True)
pdata_6mo_snpc_filter = pdata_6mo_snpc_filter.filter_sample(min_prot=1000)

pdata_6mo_snpc_norm = pdata_6mo_snpc_filter.copy()
pdata_6mo_snpc_norm.normalize(method='directlfq')

pdata_6mo_snpc_norm.summary['sample'].value_counts()

In [ ]:
pdata_fig2_region_filter = pdata_all_region_filtered.filter_prot_significant()
pdata_fig2_region_filter = pdata_fig2_region_filter.filter_sample(condition="Grouping != 'ast'")
pdata_fig2_region_filter = pdata_fig2_region_filter.filter_sample(min_prot=1000)

pdata_fig2_region_filter.summary["region"] = pdata_fig2_region_filter.summary["Grouping"].replace({
    "snpc": "SNpc",
    "ctx": "Cortex"
})
pdata_fig2_region_filter.update_summary()

pdata_fig2_region_processed = pdata_fig2_region_filter.copy()
pdata_fig2_region_processed = pdata_fig2_region_processed.filter_prot_found(min_ratio=0.4, group = ['Grouping'], match_any=True)
pdata_fig2_region_processed = pdata_fig2_region_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig2_region_norm = pdata_fig2_region_processed.copy()
pdata_fig2_region_norm.normalize(method='directlfq')

In [ ]:
pdata_fig3_snpc_filter = pdata_all_region_filtered.filter_sample(condition ='Grouping == "snpc"')
pdata_fig3_snpc_filter = pdata_fig3_snpc_filter.filter_sample(condition = 'Batch == "Apr-25"')
pdata_fig3_snpc_filter = pdata_fig3_snpc_filter.filter_prot_significant()
pdata_fig3_snpc_filter = pdata_fig3_snpc_filter.filter_sample(min_prot=1000)

pdata_fig3_snpc_filter.summary["cell_type"] = pdata_fig3_snpc_filter.summary["Sub grouping"].replace({
    "anxaY": "Anxa+",
    "anxaN": "Anxa-"
})
pdata_fig3_snpc_filter.update_summary()

pdata_fig3_snpc_processed = pdata_fig3_snpc_filter.copy()
pdata_fig3_snpc_processed = pdata_fig3_snpc_processed.filter_prot_found(min_ratio=0.4, group = ['Sub grouping'], match_any=True)
pdata_fig3_snpc_processed = pdata_fig3_snpc_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig3_snpc_norm = pdata_fig3_snpc_processed.copy()
pdata_fig3_snpc_norm.normalize(method='directlfq')

In [ ]:
# take subset of data from pdata_all_region_filtered

pdata_fig4_region_filtered = pdata_all_region_filtered.filter_prot_significant()
pdata_fig4_region_filtered = pdata_fig4_region_filtered.filter_sample(condition="Grouping != 'snpc'")
pdata_fig4_region_filtered = pdata_fig4_region_filtered.filter_sample(condition="Batch != 'Apr-25'")

mapping = {
    "ctx": "Neuron",
    "ast": "Astrocyte",
}
pdata_fig4_region_filtered.summary["cell_type"] = pdata_fig4_region_filtered.summary["Grouping"].replace(mapping)
pdata_fig4_region_filtered.update_summary()

pdata_fig4_region_processed = pdata_fig4_region_filtered.copy()
pdata_fig4_region_processed = pdata_fig4_region_processed.filter_sample(min_prot=1800)

pdata_fig4_region_processed = pdata_fig4_region_processed.filter_prot_found(min_ratio=0.4, group = ['cell_type'], match_any=True)
pdata_fig4_region_processed = pdata_fig4_region_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig4_region_norm = pdata_fig4_region_processed.copy()
pdata_fig4_region_norm.normalize(method='directlfq')

print(pdata_fig4_region_norm.summary['Grouping'].value_counts())
print(pdata_fig4_region_norm.summary['Batch'].value_counts())
print(pdata_fig4_region_norm.summary['cell_type'].value_counts())

In [ ]:
obs_columns = ['date', 'acquisition', 'size', 'cell_type', 'replicate']

pdata_fig4_swi = pAnnData.import_data(source_type='diann', report_file = 'data/2511_ast-SWI-only/report.parquet', obs_columns=obs_columns)
pdata_fig4_swi_filtered = pdata_fig4_swi.filter_sample(condition="cell_type not in ['T2','TEAB35']")
pdata_fig4_swi_filtered = pdata_fig4_swi_filtered.filter_prot_significant()
pdata_fig4_swi_filtered = pdata_fig4_swi_filtered.filter_sample(min_prot=1000)

mapping = {
    "scartissue": "SWI",
    "astrocyte": "Uninjured",
}
pdata_fig4_swi_filtered.summary["condition"] = pdata_fig4_swi_filtered.summary["cell_type"].replace(mapping)
pdata_fig4_swi_filtered.update_summary()

pdata_fig4_swi_processed = pdata_fig4_swi_filtered.copy()
pdata_fig4_swi_processed = pdata_fig4_swi_processed.filter_prot_found(min_ratio=0.4, group = ['condition'], match_any=True)
pdata_fig4_swi_processed = pdata_fig4_swi_processed.filter_prot(valid_genes=True, unique_profiles=True)

pdata_fig4_swi_norm = pdata_fig4_swi_processed.copy()
pdata_fig4_swi_norm.normalize(method='directlfq')

### analysis

In [ ]:
from scipy import stats
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.power import TTestIndPower

COMPARISONS = {
    "Agg+ vs Agg-": (
        pdata_6mo_snpc_norm,   # pdata object
        "sample",              # obs column
        "Agg+",                # group 1 label
        "Agg-",                # group 2 label
    ),
    "Anxa1+ vs Anxa1-": (
        pdata_fig3_snpc_norm,     
        "cell_type",           
        "Anxa+",              
        "Anxa-",              
    ),
    "Neuron vs Astrocyte": (
        pdata_fig4_region_norm,  
        "cell_type",           
        "Neuron",             
        "Astrocyte",          
    ),
    "SWI vs Control Astrocyte": (
        pdata_fig4_swi_norm,        
        "condition",           
        "SWI",                
        "Uninjured",           
    ),
    "Cortex vs SNpc": (
        pdata_fig2_region_norm,        
        "region",           
        "Cortex",                
        "SNpc",           
    ),
}

EFFECT_SIZES  = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
FC_GRID       = np.linspace(0.05, 2.5, 50)
N_SIM         = 5000
P_THRESH      = 0.05
LOG2FC_CUTOFF = 1.0
POWER_TARGET  = 0.80
RNG           = np.random.default_rng(42)

EQUAL_VAR = True   # pooled variance – matches the analytical power model

analysis = TTestIndPower()

# Helpers 

from dataclasses import dataclass

@dataclass
class CompVar:
    """Per-comparison variance structure for power analysis.

    Attributes:
    - sd: Pooled within-group SD, retained proteins only.
    - n1_prot: Observed count in group 1, per retained protein.
    - n2_prot: Observed count in group 2, per retained protein.
    - n1: Design size of group 1, i.e. all samples in the group.
    - n2: Design size of group 2.
    """
    sd: np.ndarray
    n1_prot: np.ndarray
    n2_prot: np.ndarray
    n1: int
    n2: int

def extract_dense(pdata_obj, sample_mask):
    """Log2 abundance matrix (n_samples, n_proteins); zeros → NaN."""
    X = pdata_obj.prot.X[sample_mask, :]
    if issparse(X):
        X = X.toarray()
    X = X.astype(float)
    X[X == 0] = np.nan
    return np.log2(X)

def pooled_within_sd(X1, X2):
    """Pooled within-group SD per protein, so between-group signal does not
    inflate sigma. Per-protein n accounts for differing missingness.

    Args:
        X1: (n1, n_proteins) log2 array for group 1.
        X2: (n2, n_proteins) log2 array for group 2.

    Returns:
        Tuple of (sd, n1, n2), each a 1-D array of length n_proteins.
        sd is NaN where either group has fewer than 2 observations.
    """
    n1 = np.sum(~np.isnan(X1), axis=0)
    n2 = np.sum(~np.isnan(X2), axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        s1 = np.nanstd(X1, axis=0, ddof=1)
        s2 = np.nanstd(X2, axis=0, ddof=1)
        num = (n1 - 1) * s1**2 + (n2 - 1) * s2**2
        sd  = np.sqrt(num / (n1 + n2 - 2))
    sd[(n1 < 2) | (n2 < 2)] = np.nan   # undefined without ≥2 obs per group
    return sd, n1, n2                  # ← was: return sd

def analytical_power(fc, sd, n1, n2, alpha=P_THRESH):
    """Two-sample t-test power at a given log2FC and SD (no fold-change filter)."""
    p = analysis.power(effect_size=fc / sd, nobs1=n1, ratio=n2 / n1, alpha=alpha)    
    if not np.isfinite(p):                      # scipy nctdtr fails at large nc
        nc = (fc / sd) * np.sqrt(n1 * n2 / (n1 + n2))
        crit = stats.t.ppf(1 - alpha / 2, n1 + n2 - 2)
        p = stats.norm.sf(crit - nc) + stats.norm.cdf(-crit - nc)
    return float(np.clip(p, 0.0, 1.0))

def comparison_curves(name):
    """Median and IQR analytical power curves for one comparison, effective n."""
    c = obs_sd_per_comp[name]
    n1e, n2e = int(np.median(c.n1_prot)), int(np.median(c.n2_prot))
    sd_lo, sd_med, sd_hi = np.percentile(c.sd, [25, 50, 75])
    pw_med = [analytical_power(fc, sd_med, n1e, n2e) for fc in FC_GRID]
    pw_lo  = [analytical_power(fc, sd_hi,  n1e, n2e) for fc in FC_GRID]
    pw_hi  = [analytical_power(fc, sd_lo,  n1e, n2e) for fc in FC_GRID]
    return pw_med, pw_lo, pw_hi

def blend_white(hex_color, alpha):
    """Blend a color toward white to emulate transparency without an alpha channel."""
    rgb = np.array(mcolors.to_rgb(hex_color))
    return tuple(alpha * rgb + (1 - alpha) * np.ones(3))

# Variance extraction (per comparison) 

obs_sd_per_comp = {}

for name, (pdata_obj, col, g1, g2) in COMPARISONS.items():
    obs   = pdata_obj.prot.obs
    m1    = (obs[col] == g1).values
    m2    = (obs[col] == g2).values
    n1, n2 = int(m1.sum()), int(m2.sum())

    X1 = extract_dense(pdata_obj, m1)
    X2 = extract_dense(pdata_obj, m2)

    sds, k1, k2 = pooled_within_sd(X1, X2)
    keep = ~np.isnan(sds) & (sds > 0)
    obs_sd_per_comp[name] = CompVar(sds[keep], k1[keep], k2[keep], n1, n2)

    c = obs_sd_per_comp[name]
    print(f"{name} (n={n1} vs {n2}): {len(c.sd)} proteins with valid within-group SD  "
          f"| median SD = {np.median(c.sd):.3f}  "
          f"| IQR = [{np.percentile(c.sd, 25):.3f}, {np.percentile(c.sd, 75):.3f}]  "
          f"| median obs = {np.median(c.n1_prot):.0f}/{n1}, {np.median(c.n2_prot):.0f}/{n2}")
    
# Simulation-based power (full DE decision rule)

power_results = {}   # {comparison_name: {effect_size: power}}

for name in COMPARISONS:
    c = obs_sd_per_comp[name]
    comp_powers = {}

    for es in EFFECT_SIZES:
        n_sig = 0
        idx = RNG.integers(len(c.sd), size=N_SIM)   # one draw → (sd, n1, n2) together
        for j in idx:
            sigma = c.sd[j]
            sim1  = RNG.normal(loc=es,  scale=sigma, size=c.n1_prot[j])
            sim2  = RNG.normal(loc=0.0, scale=sigma, size=c.n2_prot[j])

            _, p = stats.ttest_ind(sim1, sim2, equal_var=EQUAL_VAR)
            lfc  = np.mean(sim1) - np.mean(sim2)
            if p < P_THRESH and abs(lfc) >= LOG2FC_CUTOFF:
                n_sig += 1

        comp_powers[es] = n_sig / N_SIM

    power_results[name] = comp_powers

    print(f"\n{name}")
    for es, pw in comp_powers.items():
        print(f"  log2FC = {es:.2f}  →  power = {pw:.3f}")

# Analytical MDES table (t-test only)
# MDES is optimistic relative to the simulation (no log2fc)

mdes_rows = []

for name in COMPARISONS:
    c = obs_sd_per_comp[name]
    n1e, n2e = int(np.median(c.n1_prot)), int(np.median(c.n2_prot))
    d = analysis.solve_power(
        effect_size=None, nobs1=n1e, ratio=n2e / n1e,
        alpha=P_THRESH, power=POWER_TARGET,
    )
    for pct, label in [(25, "25th pct SD"), (50, "Median SD"), (75, "75th pct SD")]:
        sd_val = np.percentile(c.sd, pct)
        mdes_rows.append({
            "Comparison": name,
            "n1": n1e, "n2": n2e,                    
            "SD percentile": label,
            "SD value": round(sd_val, 3),
            "Cohen's d": round(d, 3),
            "MDES (log2FC)": round(d * sd_val, 3),
        })

mdes_df = pd.DataFrame(mdes_rows)
print("\n── Analytical MDES table ──")
print(mdes_df.to_string(index=False))

# ---------
# figures

CURVE_COLORS = {
    "Agg+ vs Agg-":             "#C64D4A",
    "Anxa1+ vs Anxa1-":         "#6F9A25",
    "Neuron vs Astrocyte":      "#A5619B",
    "SWI vs Control Astrocyte": "#1F98B1",
    "Cortex vs SNpc":           "#4E9877",
}

# Simulated power curves (full DE rule)
fig_a, ax_a = plt.subplots(figsize=(3.46, 3))

for name, comp_powers in power_results.items():
    c = obs_sd_per_comp[name]
    ax_a.plot(list(comp_powers.keys()), list(comp_powers.values()),
              marker="o", color=CURVE_COLORS[name],
              label=f"{name} (n={c.n1}, {c.n2})")

ax_a.axhline(POWER_TARGET, color="black", linestyle="--", linewidth=1,
             label=f"{int(POWER_TARGET*100)}% power")
ax_a.axvline(LOG2FC_CUTOFF, color="grey", linestyle=":", linewidth=1,
             label=f"log2FC cutoff ({LOG2FC_CUTOFF})")
ax_a.set_xlabel("True log2 fold change", fontsize=12)
ax_a.set_ylabel("Simulated power", fontsize=12)
ax_a.set_ylim(0, 1.05)
ax_a.legend(fontsize=9, loc="upper left")
ax_a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig_a.savefig("PowerAnalysis_simulation_power_curves.svg",
              bbox_inches="tight")
plt.show()

# Analytical and simulation power curves 
fig_b, ax_b = plt.subplots(figsize=(3.46, 3))

for name in COMPARISONS:
    c, col = obs_sd_per_comp[name], CURVE_COLORS[name]
    pw_med, pw_lo, pw_hi = comparison_curves(name)

    ax_b.fill_between(FC_GRID, pw_lo, pw_hi, color=blend_white(col, 0.18),
                      linewidth=0, zorder=1)
    ax_b.plot(FC_GRID, pw_med, color=col, linewidth=1.8, zorder=3, label=name)

    sim = power_results[name]
    ax_b.plot(list(sim.keys()), list(sim.values()), "o--", color=col,
              markersize=4, linewidth=1, zorder=2)

ax_b.axhline(POWER_TARGET, color="black", linestyle="--", linewidth=1,
             label=f"{int(POWER_TARGET*100)}% power")
ax_b.plot([], [], "o--", color="grey", markersize=4, linewidth=1,
          label="Simulated (with log2FC filter)")
ax_b.plot([], [], color="grey", linewidth=1.8,
          label="Analytical (t-test only, median SD)")
ax_b.set_xlabel("True log2 fold change", fontsize=12)
ax_b.set_ylabel("Power", fontsize=12)
ax_b.set_ylim(0, 1.05)
ax_b.legend(fontsize=8, loc="lower right")
ax_b.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig_b.savefig("PowerAnalysis_simulation-analytical_power_curves.svg",
              bbox_inches="tight")
plt.show()

# Analytical power curves, median SD with IQR band 
fig_b, ax_b = plt.subplots(figsize=(3.46, 3))

for name in COMPARISONS:
    color = CURVE_COLORS[name]
    pw_med, pw_lo, pw_hi = comparison_curves(name)

    ax_b.plot(FC_GRID, pw_med, color=color, linewidth=1.8,
              label=f"{name} (n={n1}, {n2})")
    ax_b.fill_between(FC_GRID, pw_lo, pw_hi, color=blend_white(color, 0.18), alpha=0.13, linewidth=0)

ax_b.set_xlabel("True log2 fold change", fontsize=12)
ax_b.axvline(LOG2FC_CUTOFF, color="grey", linestyle=":", linewidth=1,
             label=f"log2FC cutoff ({LOG2FC_CUTOFF})")
ax_b.set_ylabel("Power", fontsize=12)
ax_b.set_ylim(0, 1.05)
ax_b.legend(fontsize=8, loc="lower right")
ax_b.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig_b.savefig("PowerAnalysis_analytical_power_curves.svg",
              bbox_inches="tight")
plt.show()

# ECDF of per-protein analytical power at the log2FC cutoff 
fig_c, ax_c = plt.subplots(figsize=(3.46, 3))
frac_powered = {}

for name in COMPARISONS:
    c = obs_sd_per_comp[name]
    color = CURVE_COLORS[name]
    pw = np.array([
        analytical_power(LOG2FC_CUTOFF, s, n1k, n2k)
        for s, n1k, n2k in zip(c.sd, c.n1_prot, c.n2_prot)
    ])

    xs = np.sort(pw)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax_c.step(xs, ys, where="post", color=color, linewidth=1.8,
              label=f"{name} (n={c.n1}, {c.n2})")

    frac_powered[name] = float((pw >= POWER_TARGET).mean())

ax_c.axvline(POWER_TARGET, color="black", linestyle="--", linewidth=1,
             label=f"{int(POWER_TARGET*100)}% power")
ax_c.set_xlabel(f"Per-protein power to detect log2FC = {LOG2FC_CUTOFF}", fontsize=12)
ax_c.set_ylabel("Cumulative fraction of proteins", fontsize=12)
ax_c.set_xlim(0, 1.0)
ax_c.set_ylim(0, 1.02)
ax_c.legend(fontsize=9, loc="upper left")
ax_c.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig_c.savefig("PowerAnalysis_analytical_power_ecdf.svg",
              bbox_inches="tight")
plt.show()

# summary

print("\n── summary ──")
for name, comp_powers in power_results.items():
    c = obs_sd_per_comp[name]
    powered  = [es for es, pw in comp_powers.items() if pw >= POWER_TARGET]
    mdes_sim = min(powered) if powered else "> 2.0"
    print(
        f"{name} (n={c.n1} vs {c.n2}): "
        f"simulated power ≥ 80% from log2FC ≈ {mdes_sim}  |  "
        f"power at log2FC=1.0: {comp_powers.get(1.0, float('nan')):.2f}  |  "
        f"power at log2FC=2.0: {comp_powers.get(2.0, float('nan')):.2f}  |  "
        f"{frac_powered[name]*100:.1f}% of proteins ≥80% powered (t-test only) "
        f"at log2FC=1.0"
    )

In [ ]:
# Analytical power curves, median SD with IQR band 

CURVE_COLORS = {
    "Agg+ vs Agg-":             "#C64D4A",
    "Anxa1+ vs Anxa1-":         "#13F113",
    "Neuron vs Astrocyte":      "#A5619B",
    "SWI vs Control Astrocyte": "#1F98B1",
    "Cortex vs SNpc":           "#4E9877",
}

for name in COMPARISONS:
    color = CURVE_COLORS[name]
    pw_med, pw_lo, pw_hi = comparison_curves(name)

    fig_b, ax_b = plt.subplots(figsize=(3.46, 3))
    ax_b.fill_between(FC_GRID, pw_lo, pw_hi, color=blend_white(color, 0.18), linewidth=0)
    ax_b.plot(FC_GRID, pw_med, color=color, linewidth=1.8,
              label=f"{name} (n={c.n1}, {c.n2})")

    ax_b.axvline(LOG2FC_CUTOFF, color="grey", linestyle=":", linewidth=1,
                 label=f"log2FC cutoff ({LOG2FC_CUTOFF})")
    ax_b.set_xlabel("True log2 fold change", fontsize=12)
    ax_b.set_ylabel("Power", fontsize=12)
    ax_b.set_ylim(0, 1.05)
    ax_b.legend(fontsize=8, loc="lower right", frameon=False)
    ax_b.spines[["top", "right"]].set_visible(False)
    fig_b.tight_layout()
    scplt.shift_legend(ax_b)
    fig_b.savefig(f"PowerAnalysis_analytical_power_curves_{name}.svg", bbox_inches="tight")
    plt.show()

In [ ]:
# export plotted data
rows = []
for name in COMPARISONS:
    sds, n1, n2 = obs_sd_per_comp[name]
    sd_25, sd_50, sd_75 = np.percentile(sds, [25, 50, 75])
    for fc in FC_GRID:
        for pct_label, sd_val in [("25th", sd_25), ("median", sd_50), ("75th", sd_75)]:
            rows.append({
                "comparison":    name,
                "n1":            n1,
                "n2":            n2,
                "alpha":         P_THRESH,
                "log2FC":        fc,
                "sd_percentile": pct_label,
                "sd_value":      sd_val,
                "power":         analytical_power(fc, sd_val, n1, n2),
            })

power_curve_df = pd.DataFrame(rows)
power_curve_df.to_csv(
    "PowerAnalysis_analytical_power_curve_data.csv",
    index=False,
)


# simulation curve
fig_a_rows = []
for name, comp_powers in power_results.items():
    _, n1, n2 = obs_sd_per_comp[name]
    for log2fc, power in comp_powers.items():
        fig_a_rows.append({
            "Comparison": name,
            "n1": n1,
            "n2": n2,
            "log2FC": log2fc,
            "Simulated power": power,
        })
fig_a_data = pd.DataFrame(fig_a_rows)
fig_a_data.to_csv("MANUSCRIPT_revision/REVISION_PowerAnalysis_power_curves_data.csv", index=False)

# Figure B data: median MDES bar values, in plotted order
fig_b_data = median_mdes.rename("MDES (log2FC)").reset_index()
fig_b_data.to_csv("MANUSCRIPT_revision/REVISION_PowerAnalysis_mdes_data.csv", index=False)

fig_a_data.head(), fig_b_data.head()

In [ ]:
volcano_configs = {
    name: {
        "pdata": pdata_obj,
        "case_values": [{col: g1}, {col: g2}],
    }
    for name, (pdata_obj, col, g1, g2) in COMPARISONS.items()
}

color_dict = {
    "upregulated":   "#C64D4A",
    "downregulated": "#293DF5",
    "not_significant": "#FFFFFF6A",
}

volcano_dfs = {}

for name, cfg in volcano_configs.items():
    fig, ax = plt.subplots(figsize=(4, 4))
    ax, vdf = scplt.plot_volcano(
        ax, cfg["pdata"],
        values=cfg["case_values"],
        pval=0.05,
        return_df=True,
        fold_change_mode="mean",
        color=color_dict,
        label=[0, 0],        # suppress labels — we just want the df
    )
    plt.close(fig)           # don't display the plots
    volcano_dfs[name] = vdf


# ── Summary statistics ────────────────────────────────────────────────────────

print("── DE hit summary ──\n")
for name, vdf in volcano_dfs.items():
    sig = vdf[vdf["significance"].isin(["upregulated", "downregulated"])]
    up  = (vdf["significance"] == "upregulated").sum()
    dn  = (vdf["significance"] == "downregulated").sum()
    median_lfc = sig["log2fc"].abs().median() if len(sig) > 0 else float("nan")
    pct_above_2 = (sig["log2fc"].abs() >= 2.0).mean() * 100 if len(sig) > 0 else float("nan")

    print(f"{name}")
    print(f"  Hits: {len(sig)} total ({up} up, {dn} down)")
    print(f"  Median |log2FC| of hits: {median_lfc:.2f}")
    print(f"  % hits with |log2FC| ≥ 2.0: {pct_above_2:.1f}%\n")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, (name, vdf) in enumerate(volcano_dfs.items()):
    ax = axes[i]
    sds, n1, n2 = obs_sd_per_comp[name]

    # Significant hits
    sig = vdf[vdf["significance"].isin(["upregulated", "downregulated"])]
    abs_lfc = sig["log2fc"].abs()

    # MDES (median SD, analytical)
    median_sd = np.median(sds)
    _, n1_, n2_ = obs_sd_per_comp[name]
    try:
        d = TTestIndPower().solve_power(
            effect_size=None, nobs1=n1, ratio=n2/n1,
            alpha=P_THRESH, power=POWER_TARGET,
        )
        mdes = d * median_sd
    except Exception:
        mdes = np.nan

    # 80% power threshold from simulation (interpolate crossing)
    comp_powers = power_results[name]
    xs = np.array(list(comp_powers.keys()))
    ys = np.array(list(comp_powers.values()))
    # Linear interpolation to find where power crosses 0.8
    if ys[-1] >= POWER_TARGET:
        idx = np.searchsorted(ys, POWER_TARGET)
        if idx == 0:
            power_80_lfc = xs[0]
        else:
            x0, x1 = xs[idx-1], xs[idx]
            y0, y1 = ys[idx-1], ys[idx]
            power_80_lfc = x0 + (POWER_TARGET - y0) * (x1 - x0) / (y1 - y0)
    else:
        power_80_lfc = np.nan

    # Histogram
    ax.hist(abs_lfc, bins=30, color="#4C72B0", edgecolor="white",
            linewidth=0.5, alpha=0.85, label=f"DE hits (n={len(sig)})")

    # Vertical lines
    if not np.isnan(mdes):
        ax.axvline(mdes, color="#E84545", linestyle="--", linewidth=1.5,
                   label=f"MDES = {mdes:.2f}")
    if not np.isnan(power_80_lfc):
        ax.axvline(power_80_lfc, color="#F5A623", linestyle="-.", linewidth=1.5,
                   label=f"80% power ≈ {power_80_lfc:.2f}")

    ax.set_title(name, fontsize=11)
    ax.set_xlabel("|log2FC|", fontsize=10)
    ax.set_ylabel("Number of proteins", fontsize=10)
    ax.legend(fontsize=8, framealpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Observed |log2FC| of significant hits\nrelative to detection limits",
    fontsize=13, y=1.02
)
plt.tight_layout()
fig.savefig("MANUSCRIPT_revision/REVISION_PowerAnalysis_lfc_histogram.svg", bbox_inches="tight")
plt.show()

## 3mo snpc, NHP

In [ ]:
from scipy import stats
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.stats.power import TTestIndPower

# ── 0. Configuration ──────────────────────────────────────────────────────────

# Each entry:
#   "Display name": (pdata_object, obs_column, group1_label, group2_label)
#
# n1 and n2 are inferred automatically from the pdata obs column.

COMPARISONS = {
    "6mo Agg+ vs 6mo Agg-": (
        pdata_6mo_snpc_norm,   # pdata object
        "sample",              # obs column
        "Agg+",                # group 1 label
        "Agg-",                # group 2 label
    ),
    "3mo Agg+ vs 3mo Agg-": (
        pdata_3mo_snpc_norm,     
        "sample",           
        "Agg+",              
        "Agg-",              
    ),
}

EFFECT_SIZES  = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
FC_GRID       = np.linspace(0.05, 2.5, 50)
N_SIM         = 5000
P_THRESH      = 0.05
LOG2FC_CUTOFF = 1.0
POWER_TARGET  = 0.80
RNG           = np.random.default_rng(42)

EQUAL_VAR = True   # pooled variance – matches the analytical power model

analysis = TTestIndPower()

# Helpers 

from dataclasses import dataclass

@dataclass
class CompVar:
    """Per-comparison variance structure for power analysis.

    Attributes:
    - sd: Pooled within-group SD, retained proteins only.
    - n1_prot: Observed count in group 1, per retained protein.
    - n2_prot: Observed count in group 2, per retained protein.
    - n1: Design size of group 1, i.e. all samples in the group.
    - n2: Design size of group 2.
    """
    sd: np.ndarray
    n1_prot: np.ndarray
    n2_prot: np.ndarray
    n1: int
    n2: int

def extract_dense(pdata_obj, sample_mask):
    """Log2 abundance matrix (n_samples, n_proteins); zeros → NaN."""
    X = pdata_obj.prot.X[sample_mask, :]
    if issparse(X):
        X = X.toarray()
    X = X.astype(float)
    X[X == 0] = np.nan
    return np.log2(X)

def pooled_within_sd(X1, X2):
    """Pooled within-group SD per protein, so between-group signal does not
    inflate sigma. Per-protein n accounts for differing missingness.

    Args:
        X1: (n1, n_proteins) log2 array for group 1.
        X2: (n2, n_proteins) log2 array for group 2.

    Returns:
        Tuple of (sd, n1, n2), each a 1-D array of length n_proteins.
        sd is NaN where either group has fewer than 2 observations.
    """
    n1 = np.sum(~np.isnan(X1), axis=0)
    n2 = np.sum(~np.isnan(X2), axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        s1 = np.nanstd(X1, axis=0, ddof=1)
        s2 = np.nanstd(X2, axis=0, ddof=1)
        num = (n1 - 1) * s1**2 + (n2 - 1) * s2**2
        sd  = np.sqrt(num / (n1 + n2 - 2))
    sd[(n1 < 2) | (n2 < 2)] = np.nan   # undefined without ≥2 obs per group
    return sd, n1, n2                  # ← was: return sd

def analytical_power(fc, sd, n1, n2, alpha=P_THRESH):
    """Two-sample t-test power at a given log2FC and SD (no fold-change filter)."""
    p = analysis.power(effect_size=fc / sd, nobs1=n1, ratio=n2 / n1, alpha=alpha)    
    if not np.isfinite(p):                      # scipy nctdtr fails at large nc
        nc = (fc / sd) * np.sqrt(n1 * n2 / (n1 + n2))
        crit = stats.t.ppf(1 - alpha / 2, n1 + n2 - 2)
        p = stats.norm.sf(crit - nc) + stats.norm.cdf(-crit - nc)
    return float(np.clip(p, 0.0, 1.0))

def comparison_curves(name):
    """Median and IQR analytical power curves for one comparison, effective n."""
    c = obs_sd_per_comp[name]
    n1e, n2e = int(np.median(c.n1_prot)), int(np.median(c.n2_prot))
    sd_lo, sd_med, sd_hi = np.percentile(c.sd, [25, 50, 75])
    pw_med = [analytical_power(fc, sd_med, n1e, n2e) for fc in FC_GRID]
    pw_lo  = [analytical_power(fc, sd_hi,  n1e, n2e) for fc in FC_GRID]
    pw_hi  = [analytical_power(fc, sd_lo,  n1e, n2e) for fc in FC_GRID]
    return pw_med, pw_lo, pw_hi

def blend_white(hex_color, alpha):
    """Blend a color toward white to emulate transparency without an alpha channel."""
    rgb = np.array(mcolors.to_rgb(hex_color))
    return tuple(alpha * rgb + (1 - alpha) * np.ones(3))

# Variance extraction (per comparison) 

obs_sd_per_comp = {}

for name, (pdata_obj, col, g1, g2) in COMPARISONS.items():
    obs   = pdata_obj.prot.obs
    m1    = (obs[col] == g1).values
    m2    = (obs[col] == g2).values
    n1, n2 = int(m1.sum()), int(m2.sum())

    X1 = extract_dense(pdata_obj, m1)
    X2 = extract_dense(pdata_obj, m2)

    sds, k1, k2 = pooled_within_sd(X1, X2)
    keep = ~np.isnan(sds) & (sds > 0)
    obs_sd_per_comp[name] = CompVar(sds[keep], k1[keep], k2[keep], n1, n2)

    c = obs_sd_per_comp[name]
    print(f"{name} (n={n1} vs {n2}): {len(c.sd)} proteins with valid within-group SD  "
          f"| median SD = {np.median(c.sd):.3f}  "
          f"| IQR = [{np.percentile(c.sd, 25):.3f}, {np.percentile(c.sd, 75):.3f}]  "
          f"| median obs = {np.median(c.n1_prot):.0f}/{n1}, {np.median(c.n2_prot):.0f}/{n2}")
    
# Simulation-based power (full DE decision rule)

power_results = {}   # {comparison_name: {effect_size: power}}

for name in COMPARISONS:
    c = obs_sd_per_comp[name]
    comp_powers = {}

    for es in EFFECT_SIZES:
        n_sig = 0
        idx = RNG.integers(len(c.sd), size=N_SIM)   # one draw → (sd, n1, n2) together
        for j in idx:
            sigma = c.sd[j]
            sim1  = RNG.normal(loc=es,  scale=sigma, size=c.n1_prot[j])
            sim2  = RNG.normal(loc=0.0, scale=sigma, size=c.n2_prot[j])

            _, p = stats.ttest_ind(sim1, sim2, equal_var=EQUAL_VAR)
            lfc  = np.mean(sim1) - np.mean(sim2)
            if p < P_THRESH and abs(lfc) >= LOG2FC_CUTOFF:
                n_sig += 1

        comp_powers[es] = n_sig / N_SIM

    power_results[name] = comp_powers

    print(f"\n{name}")
    for es, pw in comp_powers.items():
        print(f"  log2FC = {es:.2f}  →  power = {pw:.3f}")

# Analytical MDES table (t-test only)
# MDES is optimistic relative to the simulation (no log2fc)

mdes_rows = []

for name in COMPARISONS:
    c = obs_sd_per_comp[name]
    n1e, n2e = int(np.median(c.n1_prot)), int(np.median(c.n2_prot))
    d = analysis.solve_power(
        effect_size=None, nobs1=n1e, ratio=n2e / n1e,
        alpha=P_THRESH, power=POWER_TARGET,
    )
    for pct, label in [(25, "25th pct SD"), (50, "Median SD"), (75, "75th pct SD")]:
        sd_val = np.percentile(c.sd, pct)
        mdes_rows.append({
            "Comparison": name,
            "n1": n1e, "n2": n2e,                    
            "SD percentile": label,
            "SD value": round(sd_val, 3),
            "Cohen's d": round(d, 3),
            "MDES (log2FC)": round(d * sd_val, 3),
        })

mdes_df = pd.DataFrame(mdes_rows)
print("\n── Analytical MDES table ──")
print(mdes_df.to_string(index=False))

# ---------
# figures

CURVE_COLORS = {
    "6mo Agg+ vs 6mo Agg-":             "#C64D4A",
    "3mo Agg+ vs 3mo Agg-":         "#C6904A",
}

# Simulated power curves (full DE rule)
fig_a, ax_a = plt.subplots(figsize=(3.46, 3))

for name, comp_powers in power_results.items():
    c = obs_sd_per_comp[name]
    ax_a.plot(list(comp_powers.keys()), list(comp_powers.values()),
              marker="o", color=CURVE_COLORS[name],
              label=f"{name} (n={c.n1}, {c.n2})")

ax_a.axhline(POWER_TARGET, color="black", linestyle="--", linewidth=1,
             label=f"{int(POWER_TARGET*100)}% power")
ax_a.axvline(LOG2FC_CUTOFF, color="grey", linestyle=":", linewidth=1,
             label=f"log2FC cutoff ({LOG2FC_CUTOFF})")
ax_a.set_xlabel("True log2 fold change", fontsize=12)
ax_a.set_ylabel("Simulated power", fontsize=12)
ax_a.set_ylim(0, 1.05)
ax_a.legend(fontsize=9, loc="upper left")
ax_a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
# fig_a.savefig("MANUSCRIPT_revision/REVISION_FigX6mo3moSnpc_PowerAnalysis_simulation_power_curves.svg",
plt.show()

# Analytical and simulation power curves 
fig_b, ax_b = plt.subplots(figsize=(3.46, 3))

for name in COMPARISONS:
    c, col = obs_sd_per_comp[name], CURVE_COLORS[name]
    pw_med, pw_lo, pw_hi = comparison_curves(name)

    ax_b.fill_between(FC_GRID, pw_lo, pw_hi, color=blend_white(col, 0.18),
                      linewidth=0, zorder=1)
    ax_b.plot(FC_GRID, pw_med, color=col, linewidth=1.8, zorder=3, label=name)

    sim = power_results[name]
    ax_b.plot(list(sim.keys()), list(sim.values()), "o--", color=col,
              markersize=4, linewidth=1, zorder=2)

ax_b.axhline(POWER_TARGET, color="black", linestyle="--", linewidth=1,
             label=f"{int(POWER_TARGET*100)}% power")
ax_b.plot([], [], "o--", color="grey", markersize=4, linewidth=1,
          label="Simulated (with log2FC filter)")
ax_b.plot([], [], color="grey", linewidth=1.8,
          label="Analytical (t-test only, median SD)")
ax_b.set_xlabel("True log2 fold change", fontsize=12)
ax_b.set_ylabel("Power", fontsize=12)
ax_b.set_ylim(0, 1.05)
ax_b.legend(fontsize=8, loc="lower right")
ax_b.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
# fig_b.savefig("MANUSCRIPT_revision/REVISION_FigX6mo3moSnpc_PowerAnalysis_simulation-analytical_power_curves.svg",
plt.show()

# Analytical power curves, median SD with IQR band 
fig_b, ax_b = plt.subplots(figsize=(3.46, 3))

for name in COMPARISONS:
    color = CURVE_COLORS[name]
    pw_med, pw_lo, pw_hi = comparison_curves(name)

    ax_b.plot(FC_GRID, pw_med, color=color, linewidth=1.8,
              label=f"{name} (n={n1}, {n2})")
    ax_b.fill_between(FC_GRID, pw_lo, pw_hi, color=blend_white(color, 0.18), alpha=0.13, linewidth=0)

ax_b.set_xlabel("True log2 fold change", fontsize=12)
ax_b.axvline(LOG2FC_CUTOFF, color="grey", linestyle=":", linewidth=1,
             label=f"log2FC cutoff ({LOG2FC_CUTOFF})")
ax_b.set_ylabel("Power", fontsize=12)
ax_b.set_ylim(0, 1.05)
ax_b.legend(fontsize=8, loc="lower right")
ax_b.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
# fig_b.savefig("MANUSCRIPT_revision/REVISION_FigX6mo3moSnpc_PowerAnalysis_analytical_power_curves.svg",
plt.show()

# ECDF of per-protein analytical power at the log2FC cutoff 
fig_c, ax_c = plt.subplots(figsize=(3.46, 3))
frac_powered = {}

for name in COMPARISONS:
    c = obs_sd_per_comp[name]
    color = CURVE_COLORS[name]
    pw = np.array([
        analytical_power(LOG2FC_CUTOFF, s, n1k, n2k)
        for s, n1k, n2k in zip(c.sd, c.n1_prot, c.n2_prot)
    ])

    xs = np.sort(pw)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax_c.step(xs, ys, where="post", color=color, linewidth=1.8,
              label=f"{name} (n={c.n1}, {c.n2})")

    frac_powered[name] = float((pw >= POWER_TARGET).mean())

ax_c.axvline(POWER_TARGET, color="black", linestyle="--", linewidth=1,
             label=f"{int(POWER_TARGET*100)}% power")
ax_c.set_xlabel(f"Per-protein power to detect log2FC = {LOG2FC_CUTOFF}", fontsize=12)
ax_c.set_ylabel("Cumulative fraction of proteins", fontsize=12)
ax_c.set_xlim(0, 1.0)
ax_c.set_ylim(0, 1.02)
ax_c.legend(fontsize=9, loc="upper left")
ax_c.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
# fig_c.savefig("MANUSCRIPT_revision/REVISION_FigX6mo3moSnpc_PowerAnalysis_analytical_power_ecdf.svg",
plt.show()

# summary

print("\n── summary ──")
for name, comp_powers in power_results.items():
    c = obs_sd_per_comp[name]
    powered  = [es for es, pw in comp_powers.items() if pw >= POWER_TARGET]
    mdes_sim = min(powered) if powered else "> 2.0"
    print(
        f"{name} (n={c.n1} vs {c.n2}): "
        f"simulated power ≥ 80% from log2FC ≈ {mdes_sim}  |  "
        f"power at log2FC=1.0: {comp_powers.get(1.0, float('nan')):.2f}  |  "
        f"power at log2FC=2.0: {comp_powers.get(2.0, float('nan')):.2f}  |  "
        f"{frac_powered[name]*100:.1f}% of proteins ≥80% powered (t-test only) "
        f"at log2FC=1.0"
    )

In [ ]:
# 20240902_Aur60minDIA_5k_TEAB35_D17

obs_columns = ['date', 'acquisition', 'sample_id','size','confirmation','thickness','type','organism','region','well_position']

pdata = pAnnData.import_data(source_type='diann',report_file = 'data/2509_human/251013_CtrlAggN/report.parquet', obs_columns=obs_columns)

# Fig X - NHP

In [ ]:
obs_columns = ['date', 'acquisition', 'sample_id','size','confirmation','thickness','type','organism','region','well_position']

pdata = pAnnData.import_data(source_type='diann',report_file = 'data/2509_NHP/human_fasta/report.parquet', obs_columns=obs_columns)

In [ ]:
pd.DataFrame(pdata.summary)

In [ ]:
pdata_nhp = pdata.filter_prot_significant()
pdata_nhp = pdata_nhp.filter_sample(min_prot=1000)

pdata_nhp.summary["type"] = pdata_nhp.summary["type"].map({"aggY": "Agg+", "aggN": "Agg-"})
pdata_nhp.update_summary()


pdata_nhp = pdata_nhp.filter_prot_found(min_ratio=0.4, group=['type'],match_any=True)
pdata_nhp = pdata_nhp.filter_prot(valid_genes=True, unique_profiles=True)

In [ ]:
pdata_nhp_norm = pdata_nhp.copy()
pdata_nhp_norm.normalize(method='directlfq')

In [ ]:
pdata_nhp_norm.export_layer("X", f"MANUSCRIPT_revision/REVISION_FigX-NHP_normalized.csv", transpose=True, var_names='Genes', obs_names=['type','sample_id'])

In [ ]:
case_values = [{'type': 'Agg+'}, {'type': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_nhp_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True,
                                    fold_change_mode='mean', color=color_dict, label=[10,10])

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_FigX-NHP_VOLCANO-directlfq.svg", bbox_inches="tight", pad_inches=0.1)
volcano_df.to_csv("MANUSCRIPT_revision/REVISION_FigX-NHP-DE-directlfq.csv")

In [ ]:
case_values = [{'type': 'Agg+'}, {'type': 'Agg-'}]

color_dict={'upregulated':"#C64D4A", 'downregulated': "#293DF5",'not_significant': "#FFFFFF6A"}

fig, ax = plt.subplots(figsize=(4, 4))
ax, volcano_df = scplt.plot_volcano(ax, pdata_nhp_norm,
                                    values=case_values,
                                    pval=0.05, return_df=True,
                                    correct_fdr=True,
                                    fold_change_mode='mean', color=color_dict, label=[10,10])

ax.set_ylabel('$log_{10}$ p value',fontsize=13, labelpad=4)
ax.set_xlabel('$log_{2}$ fold change', fontsize=13, labelpad=4)
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=11)

plt.savefig("MANUSCRIPT_revision/REVISION_FigX-NHP_VOLCANO-directlfq-adjP.svg", bbox_inches="tight", pad_inches=0.1)
volcano_df.to_csv("MANUSCRIPT_revision/REVISION_FigX-NHP-DE-directlfq-adjP.csv")

In [ ]:
pdata_nhp_norm.enrichment_functional(from_de=True, de_key="Agg+ vs Agg-")

In [ ]:
pdata_nhp_norm.stats['functional']["Agg+ vs Agg-_down"]["result"]

In [ ]:
pdata_nhp_norm.plot_enrichment_svg("Agg+ vs Agg-", direction="up")

In [ ]:
pdata_nhp_norm.plot_enrichment_svg("Agg+ vs Agg-", direction="down")

## CC

In [ ]:
import re

# (plasma membrane separate; organelle membranes go with organelle)
selected_components = [
    "cytoplasm",
    "plasma membrane",
    "extracellular region",
    "mitochondrion",
    "nucleus",
    "endoplasmic reticulum",
    "golgi apparatus",
    "cytoskeleton",
]

label_map = {
    "cytoplasm": "Cytoplasm",
    "plasma membrane": "Plasma membrane",
    "extracellular region": "Extracellular region",
    "mitochondrion": "Mitochondria",
    "nucleus": "Nucleus",
    "endoplasmic reticulum": "ER",
    "golgi apparatus": "Golgi apparatus",
    "cytoskeleton": "Cytoskeleton",
    "others": "Others",
}

plot_order = selected_components + ["others"]

def _clean_cc_term(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\[.*?\]", "", s).strip()
    return s

def map_cc_term(term: str) -> str:
    t = term.lower()

    # cytoskeleton (keep separate)
    if "cytoskeleton" in t:
        return "cytoskeleton"

    # nucleus
    if re.search(r"\bnucleus\b|nucleoplasm|nuclear", t):
        return "nucleus"

    # cytoplasm (cytosol/cytoplasmic included)
    if re.search(r"cytosol|cytoplasm|cytoplasmic", t):
        return "cytoplasm"

    # extracellular
    if "extracellular" in t:
        return "extracellular region"

    # ER (includes ER membrane etc.)
    if "endoplasmic reticulum" in t:
        return "endoplasmic reticulum"

    # Golgi
    if "golgi" in t:
        return "golgi apparatus"

    # mitochondrion (includes mitochondrial membrane etc.)
    if "mitochondri" in t:
        return "mitochondrion"

    # plasma membrane (keep separate from generic "membrane")
    if "plasma membrane" in t:
        return "plasma membrane"

    # everything else
    return "others"

def build_cc_df(df, cc_col="gene_ontology_cellular_component", protein_col=None, source_name=None):
    prot = df.index.to_series() if protein_col is None else df[protein_col]
    out = df[[cc_col]].copy()
    out["protein"] = prot.values
    out = out.rename(columns={cc_col: "cc_raw"})
    if source_name is not None:
        out["source"] = source_name

    out["cc_raw"] = out["cc_raw"].astype(str).str.split(";")
    out = out.explode("cc_raw", ignore_index=True)

    out["cc_clean"] = out["cc_raw"].map(_clean_cc_term)
    out = out[out["cc_clean"].ne("") & out["cc_clean"].ne("nan")].copy()

    out["cc_mapped"] = out["cc_clean"].map(map_cc_term)
    out["in_selected"] = out["cc_mapped"].isin(selected_components)
    return out

def counts_from_cc_df(cc_df):
    return cc_df["cc_mapped"].value_counts()

In [ ]:
nhp_upset = scutils.get_upset_contents(pdata_nhp, classes = ['type'])
nhp_neg_all = scutils.get_upset_query(nhp_upset, present='Agg-', absent=[], fetch_uniprot=True)
nhp_pos_all = scutils.get_upset_query(nhp_upset, present='Agg+', absent=[], fetch_uniprot=True)

In [ ]:
nhp_pos_all.to_csv('nhp_cc_pos.csv')
nhp_neg_all.to_csv('nhp_cc_pos.csv')

In [ ]:
nhp_neg_all

In [ ]:
df_human_proteome = pd.read_csv("MANUSCRIPT_revision/uniprotkb_taxonomy_id_9606_AND_reviewed_2026_08_19.tsv", delimiter='\t')
df_human_proteome

In [ ]:
df_nhp_proteome = pd.read_csv("MANUSCRIPT_revision/uniprotkb_taxonomy_id_9541_2026_08_19.tsv", delimiter='\t')
df_nhp_proteome

In [ ]:
# check df
neg_cc_df  = build_cc_df(nhp_neg_all,  protein_col=None, source_name="neg")
pos_cc_df = build_cc_df(nhp_pos_all, protein_col=None, source_name="pos")
human_cc_df  = build_cc_df(df_human_proteome, protein_col="Entry Name",
                           cc_col="Gene Ontology (cellular component)", source_name="human")

# counts 
neg_counts  = counts_from_cc_df(neg_cc_df).reindex(plot_order, fill_value=0)
pos_counts = counts_from_cc_df(pos_cc_df).reindex(plot_order, fill_value=0)
human_counts  = counts_from_cc_df(human_cc_df).reindex(plot_order, fill_value=0)


# convert to %
neg_pct  = neg_counts  / neg_counts.sum()  * 100
pos_pct = pos_counts / pos_counts.sum() * 100
human_pct  = human_counts  / human_counts.sum()  * 100

# order top-to-bottom to match the plot
labels = plot_order[::-1]
display_labels = [label_map[l] for l in labels]
neg_pct  = neg_pct.reindex(labels)
pos_pct = pos_pct.reindex(labels)
human_pct  = human_pct.reindex(labels)

plt.rcParams.update({
    "font.size": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})


# bar chart
fig, ax = plt.subplots(figsize=(2.7, 2.7))

y = np.arange(len(labels))
h = 0.25  # height of each bar

ax.barh(y - h, neg_pct.values,   height=h, label="NHP neg",
        color=(0.80, 0.88, 0.98), edgecolor="black", linewidth=0.6)
ax.barh(y,     pos_pct.values,   height=h, label="NHP pos",
        color=(0.00, 0.45, 0.70), edgecolor="black", linewidth=0.6)
ax.barh(y + h, human_pct.values, height=h, label="Human proteome",
        color=(0.70, 0.70, 0.70), edgecolor="black", linewidth=0.6)

ax.set_yticks(y)
ax.set_yticklabels(display_labels)
ax.set_xlabel("% proteins")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", width=1.0, length=8)

ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

# bar chart
fig, ax = plt.subplots(figsize=(2.7, 2.7))

y = np.arange(len(labels))
h = 0.35

ax.barh(y - h/2, pos_pct.values,   height=h, label="NHP",
        color=(0.00, 0.45, 0.70), edgecolor="black", linewidth=0.6)
ax.barh(y + h/2, human_pct.values, height=h, label="Human proteome",
        color=(0.70, 0.70, 0.70), edgecolor="black", linewidth=0.6)

ax.set_yticks(y)
ax.set_yticklabels(display_labels)
ax.set_xlabel("% proteins")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="both", width=1.0, length=8)

ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()